# 03 Data Prep

Single source of truth for assembling the per-anchor feature table used by the whitespace model.

**Methodology toggle**
- `ANCHOR_MODE = 'existing_store'` → Option 1 Step 1 (catchment around each Gongcha store)
- `ANCHOR_MODE = 'whitespace_grid'` → Option 1 Step 2 / Option 2 (catchment around grid centers)

Catchment radius is configurable per region tier (500m / 1km / 2km).

**Split between Python and Alteryx**
- Python: ESRI `enrich()` for demographics (pop, age/sex, income, consumption), shapefile attribute joins (admin, schools), data cleaning, count aggregation from distance tables
- Alteryx: pairwise distance between anchors and POIs within 5km (one workflow, swap inputs). Already produced for GC↔SB and GC↔GC; same pattern for SC, Tully's, schools, stations

**Notebook layout**
1. Setup & configuration
2. Anchors
3. Demographics
4. Competitors
5. Shopping centers
6. Schools
7. Region & Shinkansen tags
8. Alteryx hand-off bundle
9. Assemble final feature table (after Alteryx returns)

## 1. Setup & configuration

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / 'Data' / 'Data_for_model'
OUT_TABLES = ROOT / 'output' / 'tables'
ALX_IN = ROOT / 'output' / 'alteryx_io' / 'in'
ALX_OUT = ROOT / 'output' / 'alteryx_io' / 'out'
for p in [OUT_TABLES, ALX_IN, ALX_OUT]:
    p.mkdir(parents=True, exist_ok=True)

In [2]:
# Config
ANCHOR_MODE = 'whitespace_supplement'  # 'existing_store' | 'whitespace_grid' | 'whitespace_sc' | 'whitespace_supplement'

# Anchor filters (existing-store mode)
MIN_OPEN_YEARS = 1.0  # drop stores that have been open less than this many years

# Catchment definition (NEW methodology):
#   Square centered on the anchor, side = CATCHMENT_SIDE_M (full side length in meters).
#   500m -> the store extends 250m N/S/E/W -> 500m x 500m square.
#   Change CATCHMENT_SIDE_M to swap to a different size later (e.g. 250, 1000).
CATCHMENT_SHAPE = 'square'      # 'square' (current methodology) | 'circle' (legacy)
CATCHMENT_SIDE_M = 500          # full side length of the square in meters
CATCHMENT_HALF_M = CATCHMENT_SIDE_M / 2  # half-extent (auto-derived)

# Demographics source. ESRI is primary; WorldPop kept as sanity-check fallback.
POP_SOURCE = 'esri'  # 'esri' | 'worldpop'

# ESRI auth — points to a yaml with keys esri.username and esri.password.
ESRI_CREDS_PATH = ROOT / 'credentials.yaml'
# Existing-store mode keeps small batches (a few hundred anchors total).
# Whitespace modes scale to tens of thousands of anchors -> use larger batches to cut round-trips.
ESRI_ENRICH_BATCH_SIZE = 50 if ANCHOR_MODE.startswith('whitespace') else 5

# Whitespace candidate config.
# whitespace_grid: 500m x 500m fishnet over Japan (or selected prefectures if list is not None)
# whitespace_sc  : unopened shopping centers, one candidate per SC
GRID_SIZE_M = 500
GRID_FOCUS_PREFECTURES = None  # None = all Japan; e.g. ['Tokyo', 'Osaka'] for a smaller run
BUILD_WHITESPACE_GRID = (ANCHOR_MODE == 'whitespace_grid')
BUILD_WHITESPACE_SC = (ANCHOR_MODE == 'whitespace_sc')
BUILD_WHITESPACE_SUPPLEMENT = (ANCHOR_MODE == 'whitespace_supplement')

# Whitespace pre-filter: drop grid cells whose surrounding MLIT 1km foot-traffic mesh
# has too little presence to be commercially viable. Two layers run before ESRI enrich:
#   1) ADM1 polygon  — drops ocean cells (centers off land)
#   2) Foot traffic  — drops mountainous / rural cells (mesh pop_2021_all_alltime < thr)
#
# ESRI charges 0.01 credits per anchor x variable. With 10 model vars:
#   thr=  100 -> ~297k cells -> ~29,700 credits ($4,158)  ~8hr enrich
#   thr=  500 -> ~144k cells -> ~14,400 credits ($2,016)  ~4hr enrich  -- captures small towns
#   thr= 1000 -> ~ 95k cells -> ~ 9,500 credits ($1,330)  ~2.5hr enrich -- city + main suburbs
#   thr= 2000 -> ~ 60k cells -> ~ 6,000 credits ($  840)  ~1.7hr enrich -- dense cores only
# (pop_2021_all_alltime quantiles: p50=72, p90=1762, p95=3984)
WHITESPACE_MIN_MESH_POP = 500

# Exclusion rule: candidate center must NOT fall inside any existing Gongcha 500m x 500m square.
# This matches the existing-store catchment definition exactly (centered on the existing store).
EXCLUDE_EXISTING_GONGCHA_WITHIN_STORE_SQUARE = True

# Mode-specific output names; candidate modes do not overwrite training files.
FEATURE_FILE_STEM = {
    'existing_store': 'model_features',
    'whitespace_grid': 'model_features_whitespace_grid',
    'whitespace_sc': 'model_features_whitespace_sc',
    'whitespace_supplement': 'model_features_whitespace_supplement',
}[ANCHOR_MODE]
FEATURE_PATH = OUT_TABLES / f'{FEATURE_FILE_STEM}.csv'
FEATURE_FULL_PATH = OUT_TABLES / f'{FEATURE_FILE_STEM}_full.csv'

# Efficiency: whitespace scoring only needs the features saved in the 04 model bundle.
# If the bundle exists, candidate modes skip unused ring / nearest / legacy columns.
MODEL_FEATURES_ONLY_FOR_WHITESPACE = ANCHOR_MODE.startswith('whitespace')
MODEL_BUNDLE_PATH = ROOT / 'output' / 'models' / 'xgb_l6m_sales_model_bundle.pkl'
REQUIRED_DEPLOYMENT_FEATURES = None
if MODEL_FEATURES_ONLY_FOR_WHITESPACE and MODEL_BUNDLE_PATH.exists():
    import joblib
    REQUIRED_DEPLOYMENT_FEATURES = set(joblib.load(MODEL_BUNDLE_PATH)['feature_cols'])

print('ANCHOR_MODE       :', ANCHOR_MODE)
print('MIN_OPEN_YEARS    :', MIN_OPEN_YEARS)
print('POP_SOURCE        :', POP_SOURCE)
print('Catchment shape   :', CATCHMENT_SHAPE)
print(f'Catchment size    : {CATCHMENT_SIDE_M}m side ({CATCHMENT_HALF_M:.0f}m half-extent)')
print('Feature outputs   :', FEATURE_PATH.name, '|', FEATURE_FULL_PATH.name)
print('Model-only X mode :', MODEL_FEATURES_ONLY_FOR_WHITESPACE, '| required cols:', 0 if REQUIRED_DEPLOYMENT_FEATURES is None else len(REQUIRED_DEPLOYMENT_FEATURES))
print('ESRI creds path   :', ESRI_CREDS_PATH, '| exists:', ESRI_CREDS_PATH.exists())

ANCHOR_MODE       : whitespace_sc
MIN_OPEN_YEARS    : 1.0
POP_SOURCE        : esri
Catchment shape   : square
Catchment size    : 500m side (250m half-extent)
Feature outputs   : model_features_whitespace_sc.csv | model_features_whitespace_sc_full.csv
Model-only X mode : True | required cols: 36
ESRI creds path   : c:\Users\52333\OneDrive - Bain\Desktop\CSE\T5HM\credentials.yaml | exists: True


In [3]:
# One-time dependency installer for this kernel's environment (.venv).
# Uses `%pip` magic so packages land in the *current* kernel, not the system Python.
# Flip INSTALL_DEPS to True on a fresh machine, run this cell once, then flip back to False.
INSTALL_DEPS = False

if INSTALL_DEPS:
    req = ROOT / 'requirements.txt'
    assert req.exists(), f'requirements.txt not found at {req}'
    print(f'Installing into kernel: this can take 3-5 minutes (arcgis stack is heavy)')
    get_ipython().run_line_magic('pip', f'install -r "{req}"')

# Sanity check — imports we rely on later. If any of these fail, set INSTALL_DEPS=True and re-run.
import importlib, sys
REQUIRED = {
    'pandas': 'pandas', 'numpy': 'numpy', 'yaml': 'pyyaml', 'tqdm': 'tqdm',
    'geopandas': 'geopandas', 'shapely': 'shapely', 'pyproj': 'pyproj',
    'pyogrio': 'pyogrio',
    'arcgis': 'arcgis',
}
missing = []
for mod, pkg in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)
print('Python:', sys.executable)
if missing:
    print('MISSING packages — set INSTALL_DEPS=True and re-run this cell:', missing)
else:
    print('All required packages importable.')

Python: c:\Users\52333\OneDrive - Bain\Desktop\CSE\T5HM\.venv\Scripts\python.exe
All required packages importable.


In [4]:
# Shared helpers
def to_numeric_safe(s):
    return pd.to_numeric(s, errors='coerce')

def require_columns(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f'{name} missing required columns: {missing}')

def sanitize_text(s):
    return s.astype(str).str.replace(r'[\r\n\t]+', ' ', regex=True).str.replace('"', "'", regex=False).str.strip()

STD_POI_COLS = ['poi_id', 'poi_source', 'poi_type', 'poi_brand', 'name', 'address', 'lat', 'lng']

def to_std_poi(df, *, poi_source, poi_type, poi_brand=None, id_col=None, name_col=None, addr_col=None,
               lat_col='lat', lng_col='lng', extra_cols=None):
    """Coerce a dataframe into the standardized POI schema.
    extra_cols: list of additional attribute columns to carry through."""
    out = pd.DataFrame()
    out['poi_id'] = df[id_col].astype(str) if id_col else (poi_source + '_' + df.index.astype(str))
    out['poi_source'] = poi_source
    out['poi_type'] = poi_type
    out['poi_brand'] = poi_brand if poi_brand is not None else df.get('brand', pd.NA)
    out['name'] = df[name_col] if name_col else pd.NA
    out['address'] = df[addr_col] if addr_col else pd.NA
    out['lat'] = to_numeric_safe(df[lat_col])
    out['lng'] = to_numeric_safe(df[lng_col])
    if extra_cols:
        for c in extra_cols:
            out[c] = df[c] if c in df.columns else pd.NA
    return out

def csv_write(df, path, sanitize_cols=None):
    import csv
    if sanitize_cols:
        for c in sanitize_cols:
            if c in df.columns:
                df[c] = sanitize_text(df[c])
    df.to_csv(path, index=False, encoding='utf-8-sig', lineterminator='\n', quoting=csv.QUOTE_MINIMAL)
    print('Wrote', path, '|', len(df), 'rows,', len(df.columns), 'cols')

## 2. Anchors

Anchor = the point around which a catchment is drawn.
- 2a — existing-store anchors come from Gongcha cleaned table (reuse `adhoc_gongcha_points.csv`)
- 2b — whitespace-grid anchors come from a fishnet over focus prefectures, dropping cells containing a Gongcha store

In [5]:
# 2a. Existing-store anchors (Gongcha, currently open, tenure >= MIN_OPEN_YEARS)
gc = pd.read_csv(OUT_TABLES / 'adhoc_gongcha_points.csv')
n_total = len(gc)

# Step 1: keep only open stores
gc_open = gc[gc['open_close'].astype(str).str.lower() == 'open'].copy()
n_open = len(gc_open)

# Step 2: filter out stores with tenure < MIN_OPEN_YEARS
gc_open['open_years'] = pd.to_numeric(gc_open['open_years'], errors='coerce')
gc_mature = gc_open[gc_open['open_years'] >= MIN_OPEN_YEARS].copy()
n_mature = len(gc_mature)

anchors_existing = pd.DataFrame({
    'anchor_id': 'gc_' + gc_mature['gongcha_id'].astype(str),
    'anchor_source': 'gongcha_existing',
    'anchor_name': gc_mature['gongcha_name'],
    'address': gc_mature['address'],
    'lat': gc_mature['lat'],
    'lng': gc_mature['lng'],
    'store_type': gc_mature['store_type'],
    'open_year': gc_mature['open_year'],
    'open_years': gc_mature['open_years'],
    'catchment_side_m': CATCHMENT_SIDE_M,
})
anchors_existing = anchors_existing.dropna(subset=['lat', 'lng']).reset_index(drop=True)

# Pull store-side attributes from the Gongcha xlsx (only present there, not in the cleaned csv).
#   Sq. Ft.       -> store area (square feet)
#   Seats         -> seat count
#   IS/RS         -> Inside Shopping (IS) vs Road Side (RS)  -> derive is_inside_sc flag
#   Biz Zone      -> Urban SC / Suburbs SC / Terminal / Roadside
#   Location Type -> finer-grained format (Shopping Center / Commercial building / Terminal / ...)
# Match key: No. == gongcha_id. (Store Type col is 100% 'NP' for Japan, so skipped.)
_gc_xlsx = DATA_DIR / 'internal' / '260513_global_store_address_vshare_ジオコーディング修正_手作業追加_vup.xlsx'
_attr = pd.read_excel(_gc_xlsx, sheet_name='Sheet2_Database', header=9,
                      usecols=['No.', 'Sq. Ft.', 'Seats', 'IS/RS', 'Biz Zone', 'Location Type'])
_attr = _attr.dropna(subset=['No.'])
_attr['gongcha_id']      = pd.to_numeric(_attr['No.'], errors='coerce').astype('Int64')
_attr['store_area_sqft'] = pd.to_numeric(_attr['Sq. Ft.'], errors='coerce')
_attr['store_area_m2']   = _attr['store_area_sqft'] * 0.092903   # 1 sq ft = 0.0929 m^2
_attr['seats']           = pd.to_numeric(_attr['Seats'], errors='coerce')
# Format / location flags
_attr['is_rs']         = _attr['IS/RS'].astype(str).str.strip().str.upper()
_attr['is_inside_sc'] = (_attr['is_rs'] == 'IS').astype('Int8')
_attr['biz_zone']     = _attr['Biz Zone'].astype(str).str.strip()
_attr['location_type']= _attr['Location Type'].astype(str).str.strip()
_attr = _attr[['gongcha_id', 'store_area_sqft', 'store_area_m2', 'seats',
               'is_inside_sc', 'biz_zone', 'location_type']]

# Merge by integer gongcha_id (strip the 'gc_' prefix already on anchor_id)
anchors_existing['gongcha_id'] = anchors_existing['anchor_id'].str.replace('gc_', '', regex=False).astype('Int64')
anchors_existing = anchors_existing.merge(_attr, on='gongcha_id', how='left').drop(columns=['gongcha_id'])

_n_area  = anchors_existing['store_area_m2'].notna().sum()
_n_seats = anchors_existing['seats'].notna().sum()
_n_fmt   = anchors_existing['is_inside_sc'].notna().sum()
_n       = len(anchors_existing)
print(f'Gongcha rows total      : {n_total}')
print(f'  after open filter     : {n_open}  (-{n_total - n_open} closed)')
print(f'  after tenure >= {MIN_OPEN_YEARS}y : {n_mature}  (-{n_open - n_mature} too new)')
print(f'  after geo cleanup     : {len(anchors_existing)}')
print(f'  store_area coverage   : {_n_area}/{_n} ({_n_area/_n*100:.0f}%)  '
      f'median {anchors_existing["store_area_m2"].median():.0f} m^2')
print(f'  seats coverage        : {_n_seats}/{_n} ({_n_seats/_n*100:.0f}%)  '
      f'median {anchors_existing["seats"].median():.0f} seats')
print(f'  store format coverage : {_n_fmt}/{_n} ({_n_fmt/_n*100:.0f}%)')
print()
print('Format breakdown (after tenure filter):')
print(f'  is_inside_sc  : IS={int((anchors_existing["is_inside_sc"]==1).sum())}, '
      f'RS={int((anchors_existing["is_inside_sc"]==0).sum())}')
print('  biz_zone:')
print(anchors_existing['biz_zone'].value_counts().to_string().replace('\n', '\n    '))
print('  location_type:')
print(anchors_existing['location_type'].value_counts().to_string().replace('\n', '\n    '))
anchors_existing.head(3)

Gongcha rows total      : 260
  after open filter     : 217  (-43 closed)
  after tenure >= 1.0y : 186  (-31 too new)
  after geo cleanup     : 186
  store_area coverage   : 184/186 (99%)  median 60 m^2
  seats coverage        : 124/186 (67%)  median 20 seats
  store format coverage : 186/186 (100%)

Format breakdown (after tenure filter):
  is_inside_sc  : IS=176, RS=10
  biz_zone:
biz_zone
    Suburbs SC    78
    Urban SC      58
    Terminal      27
    Roadside      23
  location_type:
location_type
    Shopping Center                     66
    Commercial building                 65
    Terminal                            28
    Street-level (roadside)             23
    NP                                   3
    Service area (highway rest area)     1


,anchor_id,anchor_source,anchor_name,address,lat,lng,store_type,open_year,open_years,catchment_side_m,store_area_sqft,store_area_m2,seats,is_inside_sc,biz_zone,location_type
0,gc_1420,gongcha_existing,ekimo Namba,"ekimo Namba, 1-9-7, Namba, Chuo-ku, Osaka-shi",34.667325,135.500732,FC,2020.0,5.757700,500,1154.377900,107.245170,16.0,1,Terminal,Terminal
1,gc_1315,gongcha_existing,LUMINE Tachikawa,"1F, LUMINE Tachikawa, 2-1-1, Akebono-cho, Tach...",35.698449,139.413761,DOS,2016.0,9.691992,500,453.864106,42.165337,NaN,1,Urban SC,Commercial building
2,gc_1377,gongcha_existing,Beans Asagaya,"1F, Beans Asagaya, 3-36-1,Asagayaminami, Sugin...",35.704839,139.636480,FC,2016.0,10.053388,500,608.737166,56.553509,22.0,1,Terminal,Terminal


In [6]:
# 2b. Whitespace anchors (build only when needed)
# Two candidate universes:
#   1) whitespace_grid: 500m x 500m fishnet over Japan; one candidate at each cell centroid
#   2) whitespace_sc  : unopened shopping centers; one candidate per SC POI
#
# Important exclusion rule for both:
#   candidate center must NOT fall inside any existing Gongcha 500m x 500m square.
#   This is stricter than just removing the exact occupied cell and matches our catchment definition.

def exclude_centers_inside_existing_store_squares(candidate_df, existing_store_df, side_m=CATCHMENT_SIDE_M):
    import geopandas as gpd
    from shapely.geometry import box
    if candidate_df is None or len(candidate_df) == 0 or existing_store_df is None or len(existing_store_df) == 0:
        return candidate_df
    half = side_m / 2
    cand = candidate_df.dropna(subset=['lat', 'lng']).copy()
    cand_g = gpd.GeoDataFrame(
        cand, geometry=gpd.points_from_xy(cand['lng'], cand['lat']), crs='EPSG:4326').to_crs(epsg=3857)
    stores = existing_store_df.dropna(subset=['lat', 'lng']).copy()
    store_g = gpd.GeoDataFrame(
        stores[['anchor_id', 'lat', 'lng']],
        geometry=gpd.points_from_xy(stores['lng'], stores['lat']), crs='EPSG:4326').to_crs(epsg=3857)
    store_g['geometry'] = store_g.geometry.apply(lambda pt: box(pt.x - half, pt.y - half, pt.x + half, pt.y + half))
    hit = gpd.sjoin(cand_g, store_g[['anchor_id', 'geometry']], how='left', predicate='within')
    excluded_idx = hit.loc[hit['index_right'].notna()].index.unique()
    out = cand_g.loc[~cand_g.index.isin(excluded_idx)].drop(columns='geometry').reset_index(drop=True)
    dropped = int(len(cand_g) - len(out))
    print(f'Excluded {dropped:,} candidates whose centers fall inside existing-store {side_m}m squares')
    return out


def _load_foot_traffic_cache_for_prefilter():
    """Load the MLIT 1km mesh cache (built in section 8c) for grid prefiltering.
    Returns None if cache not yet built — prefilter is then skipped (with a warning)."""
    ft_cache = ROOT / 'output' / 'cache' / 'foot_traffic_mesh.pkl'
    if not ft_cache.exists():
        print(f'WARN: foot traffic cache not found at {ft_cache} — grid prefilter skipped')
        return None
    return pd.read_pickle(ft_cache)


def prefilter_grid_by_foot_traffic(cells_3857_gdf, min_pop=WHITESPACE_MIN_MESH_POP):
    """Drop grid cells whose center is not within ~500m of a populated MLIT 1km mesh.

    A populated mesh is one with `pop_2021_all_alltime >= min_pop`.
    We expand each populated mesh to a 1km square in EPSG:3857 and spatial-join the
    candidate centers against the union. Returns a boolean mask aligned with cells_3857_gdf.
    """
    import geopandas as gpd
    from shapely.geometry import box
    ft = _load_foot_traffic_cache_for_prefilter()
    if ft is None:
        return None
    # Pop column used for filter — pick the latest "all_alltime" if present, else any pop_*.
    pop_col = next((c for c in ['pop_2021_all_alltime', 'pop_2019_all_alltime']
                    if c in ft.columns), None)
    if pop_col is None:
        pop_col = next((c for c in ft.columns if c.startswith('pop_')), None)
    if pop_col is None:
        print('WARN: no pop_* column in foot traffic cache — grid prefilter skipped')
        return None
    pop_meshes = ft[ft[pop_col] >= min_pop][['lat', 'lng', pop_col]].copy()
    print(f'  populated meshes ({pop_col} >= {min_pop}): {len(pop_meshes):,} / {len(ft):,}')
    if len(pop_meshes) == 0:
        return None
    # 1km squares (centered on mesh lat/lng in projected CRS).
    mesh_g = gpd.GeoDataFrame(
        pop_meshes,
        geometry=gpd.points_from_xy(pop_meshes['lng'], pop_meshes['lat']),
        crs='EPSG:4326').to_crs(epsg=3857)
    mesh_g['geometry'] = mesh_g.geometry.apply(lambda pt: box(pt.x - 500, pt.y - 500, pt.x + 500, pt.y + 500))
    hit = gpd.sjoin(cells_3857_gdf, mesh_g[['geometry']], how='left', predicate='within')
    keep_idx = hit.loc[hit['index_right'].notna()].index.unique()
    mask = cells_3857_gdf.index.isin(keep_idx)
    print(f'  kept {int(mask.sum()):,} / {len(cells_3857_gdf):,} cells after foot-traffic prefilter')
    return mask


def build_whitespace_grid(focus_prefectures=None, grid_size_m=500):
    import geopandas as gpd
    from shapely.geometry import box
    adm1 = gpd.read_file(DATA_DIR / 'admin_shp' / 'jpn_adm_2019_shp' / 'jpn_admbnda_adm1_2019.shp')
    if focus_prefectures:
        adm1 = adm1[adm1['ADM1_EN'].isin(focus_prefectures)].copy()
    adm1_m = adm1.to_crs(epsg=3857)
    minx, miny, maxx, maxy = adm1_m.total_bounds
    xs = np.arange(minx, maxx, grid_size_m)
    ys = np.arange(miny, maxy, grid_size_m)
    cells = [box(x, y, x + grid_size_m, y + grid_size_m) for x in xs for y in ys]
    grid = gpd.GeoDataFrame({'geometry': cells}, crs='EPSG:3857')
    grid['centroid_geom'] = grid.geometry.centroid
    # Keep cells whose center is inside Japan/prefecture polygon. This drops most coastline/water cells.
    cent = gpd.GeoDataFrame(grid.drop(columns='geometry'), geometry='centroid_geom', crs='EPSG:3857')
    cent = gpd.sjoin(cent, adm1_m[['ADM1_EN', 'geometry']], how='inner',
                     predicate='within').drop(columns='index_right').reset_index(drop=True)
    print(f'After ADM1 polygon filter   : {len(cent):,} cells')

    # Foot-traffic prefilter — drops mountainous / unpopulated cells before ESRI enrichment.
    if WHITESPACE_MIN_MESH_POP and WHITESPACE_MIN_MESH_POP > 0:
        mask = prefilter_grid_by_foot_traffic(cent, min_pop=WHITESPACE_MIN_MESH_POP)
        if mask is not None:
            cent = cent.loc[mask].reset_index(drop=True)
            print(f'After foot-traffic prefilter: {len(cent):,} cells')

    cent_ll = cent.geometry.to_crs(epsg=4326)
    out = pd.DataFrame({
        'anchor_id': [f'wsg_{i:08d}' for i in range(len(cent))],
        'anchor_source': 'whitespace_grid',
        'anchor_name': pd.NA,
        'address': pd.NA,
        'lat': cent_ll.y.values,
        'lng': cent_ll.x.values,
        'prefecture_en': cent['ADM1_EN'].values,
        'catchment_side_m': CATCHMENT_SIDE_M,
        'is_inside_sc': 0,  # grid points are generic; SC candidates are handled in whitespace_sc mode
    })
    if EXCLUDE_EXISTING_GONGCHA_WITHIN_STORE_SQUARE:
        out = exclude_centers_inside_existing_store_squares(out, anchors_existing, CATCHMENT_SIDE_M)
    print(f'Final whitespace grid       : {len(out):,} anchors')
    return out


def load_sc_candidates_for_whitespace():
    # Minimal SC loader duplicated here so whitespace_sc anchors exist before ESRI enrichment.
    sc_xls = pd.ExcelFile(DATA_DIR / 'shopping_center' / '260517_SC list_v1.xlsx')
    sc_db_sheet = next((s for s in sc_xls.sheet_names if 'SC' in s and ('情報' in s or '売上' in s)), sc_xls.sheet_names[1])
    sc_raw = pd.read_excel(sc_xls, sheet_name=sc_db_sheet, header=6)
    sc_raw.columns = [str(c).strip() for c in sc_raw.columns]
    sc_col_map = {
        'RecID': 'sc_id', 'SC名称': 'name', '都道府県名': 'prefecture_jp', '市区町村名': 'city_jp',
        '所在地': 'address', '店舗面積': 'sales_floor_m2', '建物延床面積': 'building_floor_m2',
        'テナント数総数': 'tenant_count', '売上高（百万円）': 'sales_mn_jpy', '営業面積(㎡)': 'op_floor_m2',
        'X': 'lng', 'Y': 'lat',
    }
    sc = sc_raw.rename(columns=sc_col_map)
    keep_cols = [v for v in sc_col_map.values() if v in sc.columns]
    sc = sc[keep_cols].copy()
    for c in ['lat', 'lng', 'sales_floor_m2', 'sales_mn_jpy']:
        if c in sc.columns:
            sc[c] = to_numeric_safe(sc[c])
    sc = sc.dropna(subset=['lat', 'lng']).reset_index(drop=True)
    out = pd.DataFrame({
        'anchor_id': 'wssc_' + sc['sc_id'].astype(str),
        'anchor_source': 'whitespace_sc',
        'anchor_name': sc['name'],
        'address': sc['address'],
        'lat': sc['lat'],
        'lng': sc['lng'],
        'catchment_side_m': CATCHMENT_SIDE_M,
        'is_inside_sc': 1,
        'candidate_sc_id': sc['sc_id'],
        'candidate_sc_sales_floor_m2': sc.get('sales_floor_m2'),
        'candidate_sc_sales_mn_jpy': sc.get('sales_mn_jpy'),
    })
    if EXCLUDE_EXISTING_GONGCHA_WITHIN_STORE_SQUARE:
        out = exclude_centers_inside_existing_store_squares(out, anchors_existing, CATCHMENT_SIDE_M)
    return out.reset_index(drop=True)

if BUILD_WHITESPACE_GRID:
    anchors_whitespace = build_whitespace_grid(GRID_FOCUS_PREFECTURES, GRID_SIZE_M)
    print('Whitespace grid anchors:', len(anchors_whitespace))
else:
    anchors_whitespace = None
    print('Skipping whitespace grid (flag off)')

if BUILD_WHITESPACE_SC:
    anchors_whitespace_sc = load_sc_candidates_for_whitespace()
    print('Whitespace SC anchors:', len(anchors_whitespace_sc))
else:
    anchors_whitespace_sc = None
    print('Skipping whitespace SC candidates (flag off)')

def build_whitespace_supplement():
    """v4 supplement anchors: 889 lattice-snapped grid cells covering missing SCs
    + 161 SC anchors (v3 universe minus v1-enriched)."""
    g = pd.read_csv(OUT_TABLES / 'new_grid_cells_to_enrich.csv', encoding='utf-8-sig')
    g_out = pd.DataFrame({
        'anchor_id': g['new_anchor_id'].astype(str),
        'anchor_source': 'whitespace_grid_supplement',
        'anchor_name': g['new_anchor_id'].astype(str),
        'address': '',
        'lat': g['centroid_lat'].astype(float),
        'lng': g['centroid_lng'].astype(float),
        'catchment_side_m': CATCHMENT_SIDE_M,
        'is_inside_sc': 0,
    })
    s = pd.read_csv(OUT_TABLES / 'sc_needs_enrichment.csv', encoding='utf-8-sig')
    s_out = pd.DataFrame({
        'anchor_id': s['anchor_id'].astype(str),
        'anchor_source': 'whitespace_sc_supplement',
        'anchor_name': s['sc_name'].astype(str),
        'address': '',
        'lat': s['lat'].astype(float),
        'lng': s['lng'].astype(float),
        'catchment_side_m': CATCHMENT_SIDE_M,
        'is_inside_sc': 1,
    })
    return pd.concat([g_out, s_out], ignore_index=True)

if BUILD_WHITESPACE_SUPPLEMENT:
    anchors_supplement = build_whitespace_supplement()
    print(f'Whitespace supplement anchors: {len(anchors_supplement)} '
          f'({(anchors_supplement.anchor_source=="whitespace_grid_supplement").sum()} grid + '
          f'{(anchors_supplement.anchor_source=="whitespace_sc_supplement").sum()} sc)')
else:
    anchors_supplement = None
    print('Skipping whitespace supplement anchors (flag off)')

Skipping whitespace grid (flag off)
Excluded 307 candidates whose centers fall inside existing-store 500m squares
Whitespace SC anchors: 2786


In [7]:
# Pick the active anchor set based on ANCHOR_MODE
if ANCHOR_MODE == 'existing_store':
    anchors = anchors_existing.copy()
elif ANCHOR_MODE == 'whitespace_grid':
    assert anchors_whitespace is not None, 'Set ANCHOR_MODE="whitespace_grid" and re-run cell 2b first.'
    anchors = anchors_whitespace.copy()
elif ANCHOR_MODE == 'whitespace_sc':
    assert anchors_whitespace_sc is not None, 'Set ANCHOR_MODE="whitespace_sc" and re-run cell 2b first.'
    anchors = anchors_whitespace_sc.copy()
elif ANCHOR_MODE == 'whitespace_supplement':
    assert anchors_supplement is not None, 'Set ANCHOR_MODE="whitespace_supplement" and re-run cell 2b first.'
    anchors = anchors_supplement.copy()
else:
    raise ValueError(f'Unknown ANCHOR_MODE: {ANCHOR_MODE}')
print('Active anchor set:', ANCHOR_MODE, '|', len(anchors), 'rows')
print('Output files:', FEATURE_PATH.name, '|', FEATURE_FULL_PATH.name)
anchors.head(2)

Active anchor set: whitespace_sc | 2786 rows
Output files: model_features_whitespace_sc.csv | model_features_whitespace_sc_full.csv


,anchor_id,anchor_source,anchor_name,address,lat,lng,catchment_side_m,is_inside_sc,candidate_sc_id,candidate_sc_sales_floor_m2,candidate_sc_sales_mn_jpy
0,wssc_1101012,whitespace_sc,サッポロファクトリー,北二条東4丁目1-2,43.064941,141.362678,500,1,1101012,44631.7,14003.0
1,wssc_1101017,whitespace_sc,イオン札幌桑園ショッピングセンター,北8条西14-28,43.069413,141.333112,500,1,1101017,21581.0,NaN


## 3. Demographics — ESRI enrich

Catchment = **`CATCHMENT_SIDE_M` × `CATCHMENT_SIDE_M` square centered on the anchor**
(default 500m × 500m, i.e. ±250m N/S/E/W). Shape and side length are config-driven
(`CATCHMENT_SHAPE`, `CATCHMENT_SIDE_M` in Section 1).

Per anchor catchment we pull from ESRI Japan in **one** enrich call:
- total population
- age × sex bands (used to derive women 10-29)
- average / median household income
- beverage consumption (FoodEsriJapan C0214/C0215/C0219 — proven in the reference notebook)

Flow (mirrors `Enrich_Vars_only_Dist_Esri_vTest.ipynb`):
1. Auth via `credentials.yaml` (keys: `esri.username`, `esri.password`)
2. Build square polygons in EPSG:3857 (`shapely.geometry.box`), then reproject to EPSG:4326
3. One row per anchor (single catchment now), keyed by `anchor_id`
4. `enrich()` once over the whole study-area set
5. Join enriched columns back to anchors

Two things ESRI does **not** cover and still need MLIT:
- Foot traffic 1km grid (real footfall) → MLIT 1kmメッシュ別将来推計人口 (proxy) or paid Agoop/Docomo
- Station 乗降客数 → MLIT 駅別乗降客数 shapefile

In [8]:
# 3a. ESRI auth
# Tries 3 sources in order: credentials.yaml -> env vars -> interactive prompt
import os
import yaml
from getpass import getpass

def _load_creds(creds_path=ESRI_CREDS_PATH):
    # 1. yaml file
    if Path(creds_path).exists():
        creds = yaml.safe_load(Path(creds_path).read_text(encoding='utf-8')) or {}
        e = creds.get('esri', {})
        if e.get('username') and e.get('password'):
            return e['username'], e['password'], f'credentials.yaml ({creds_path.name})'
    # 2. env vars
    u = os.environ.get('ARCGIS_USERNAME') or os.environ.get('ESRI_USERNAME')
    p = os.environ.get('ARCGIS_PASSWORD') or os.environ.get('ESRI_PASSWORD')
    if u and p:
        return u, p, 'env vars (ARCGIS_USERNAME / ARCGIS_PASSWORD)'
    # 3. interactive
    print('No credentials.yaml or env vars found. Enter ESRI credentials:')
    u = input('ESRI username: ').strip()
    p = getpass('ESRI password: ')
    return u, p, 'interactive prompt'

def esri_login(creds_path=ESRI_CREDS_PATH):
    from arcgis.gis import GIS
    username, password, source = _load_creds(creds_path)
    if not username or not password:
        raise ValueError('ESRI username/password are empty')
    print(f'Auth source: {source}')
    return GIS(username=username, password=password, verify_cert=False)

RUN_ESRI_ENRICH = True  # set False to skip ESRI section entirely
if RUN_ESRI_ENRICH and POP_SOURCE == 'esri':
    gis_private = esri_login()
    print('Remaining credits:', gis_private.admin.credits.credits)
else:
    gis_private = None
    print('Skipping ESRI auth (RUN_ESRI_ENRICH is False or POP_SOURCE != esri)')

Setting `verify_cert` to False is a security risk, use at your own risk.


Auth source: credentials.yaml (credentials.yaml)
Remaining credits: 983402.44


In [9]:
# 3b. Build catchment polygons — one square per anchor.
# Projection: EPSG:3857 (web mercator) is fine for Japan-scale shape building.
# For a more accurate equal-area square swap to a JGD2011 zone CRS later.
def build_catchment_studyareas(anchors_df, side_m=CATCHMENT_SIDE_M, shape=CATCHMENT_SHAPE):
    """Return a GeoAccessor (spatially-enabled DataFrame) ready for enrich().
    One polygon per anchor, keyed by anchor_id. Polygons returned in EPSG:4326."""
    import geopandas as gpd
    from shapely.geometry import box
    from arcgis import features as arc_features
    half = side_m / 2
    gdf = gpd.GeoDataFrame(
        anchors_df[['anchor_id', 'lat', 'lng']].copy(),
        geometry=gpd.points_from_xy(anchors_df['lng'], anchors_df['lat']),
        crs='EPSG:4326').to_crs('EPSG:3857')
    if shape == 'square':
        gdf['geometry'] = gdf.geometry.apply(lambda p: box(p.x - half, p.y - half, p.x + half, p.y + half))
    elif shape == 'circle':
        gdf['geometry'] = gdf.geometry.buffer(half)  # half-side acts as radius
    else:
        raise ValueError(f'Unsupported CATCHMENT_SHAPE: {shape!r}')
    gdf['catchment_shape'] = shape
    gdf['catchment_side_m'] = side_m
    gdf = gdf.to_crs('EPSG:4326')
    sdf = arc_features.GeoAccessor.from_geodataframe(gdf)
    return sdf

if RUN_ESRI_ENRICH and POP_SOURCE == 'esri':
    study_areas = build_catchment_studyareas(anchors, CATCHMENT_SIDE_M, CATCHMENT_SHAPE)
    print(f'Catchment polygons built: {len(study_areas)} '
          f'({len(anchors)} anchors × 1 {CATCHMENT_SHAPE} of {CATCHMENT_SIDE_M}m side)')
else:
    study_areas = None

Catchment polygons built: 2786 (2786 anchors × 1 square of 500m side)


In [10]:
# 3c. Discover ESRI Japan variables.
# Run this cell once to find the analysisVariable IDs for population, age/sex, income, etc.
# Pattern: country.data_collections is a DataFrame indexed by dataCollectionID with columns analysisVariable, alias.
DISCOVER_VARS = True  # flip True to dump the variable catalog
if DISCOVER_VARS and RUN_ESRI_ENRICH:
    from arcgis.geoenrichment import Country
    country = Country.get('Japan', gis=gis_private)
    jp_vars = country.data_collections
    # Common buckets to inspect
    for bucket in ['KeyJapanFacts', 'AtRiskPopulationJapan', 'HousingJapan',
                   'FoodEsriJapan', 'agegender', 'incomeJapan']:
        sub = jp_vars[jp_vars.index.astype(str).str.contains(bucket, case=False, na=False)]
        if len(sub):
            print(f'\n=== {bucket} ({len(sub)} vars) ===')
            print(sub[['analysisVariable', 'alias']].head(15).to_string())
    # Also dump the full catalog to disk for offline lookup
    jp_vars.to_csv(OUT_TABLES / 'esri_japan_data_collections.csv', encoding='utf-8-sig')
    print('\nFull catalog written to esri_japan_data_collections.csv')


=== FoodEsriJapan (254 vars) ===
                     analysisVariable                                           alias
dataCollectionID                                                                     
FoodEsriJapan     FoodEsriJapan.C0002                                       2026 Food
FoodEsriJapan     FoodEsriJapan.C0003  2026 Cereals (Rice, Bread, Noodles, & Cereals)
FoodEsriJapan     FoodEsriJapan.C0004                                       2026 Rice
FoodEsriJapan     FoodEsriJapan.C0005                                      2026 Bread
FoodEsriJapan     FoodEsriJapan.C0006                         2026 Bread: White Bread
FoodEsriJapan     FoodEsriJapan.C0007                         2026 Bread: Other Bread
FoodEsriJapan     FoodEsriJapan.C0008                                    2026 Noodles
FoodEsriJapan     FoodEsriJapan.C0009               2026 Noodles: Non-Dried Udon/Soba
FoodEsriJapan     FoodEsriJapan.C0010                   2026 Noodles: Dried Udon/Soba
FoodEsriJapan     Fo

In [11]:
# 3d. Variable selection.
#
# Year rule: prefer 2025 (most recent estimate available).
# Beverage spending only has 2026 vintage in ESRI Japan -> use 2026.
# Population/income totals: 2025 (PopulationTotalsEsriJapan / HouseholdsbyIncomeEsriJapan).
# All IDs below were verified against esri_japan_data_collections.csv (cell 3c output).
ENRICH_VAR_MAP = {
    # --- Population totals (2025) ---
    'pop_total':            'PopulationTotalsEsriJapan.F0103',   # 2025 Total Population
    'pop_female_total':     'PopulationTotalsEsriJapan.F0137',   # 2025 Female Population

    # --- Households & income (2025) ---
    'households_total':     'HouseholdsbyIncomeEsriJapan.I_BASE',        # 2025 Total Households
    'household_income_avg': 'HouseholdsbyIncomeEsriJapan.I_AVE_GENHHHO', # 2025 Avg HH Income

    # --- Beverage spending (2026, FoodEsriJapan) ---
    'beverages_total':      'FoodEsriJapan.C0214',  # 2026 Beverages
    'beverages_tea':        'FoodEsriJapan.C0215',  # 2026 Beverages/Tea
    'beverages_tea_drinks': 'FoodEsriJapan.C0219',  # 2026 Beverages/Tea: Tea Beverages

    # --- 2025 Total population by 5-year age band (5YearIncrementsEsriJapan, F0106-F0110) ---
    'pop_age_10_14':        '5YearIncrementsEsriJapan.F0106',
    'pop_age_15_19':        '5YearIncrementsEsriJapan.F0107',
    'pop_age_20_24':        '5YearIncrementsEsriJapan.F0108',
    'pop_age_25_29':        '5YearIncrementsEsriJapan.F0109',
    'pop_age_30_34':        '5YearIncrementsEsriJapan.F0110',

    # --- 2025 Female population by 5-year age band (F0140-F0144). cell 3f sums 10-29 -> women_10_29. ---
    'female_10_14':         '5YearIncrementsEsriJapan.F0140',
    'female_15_19':         '5YearIncrementsEsriJapan.F0141',
    'female_20_24':         '5YearIncrementsEsriJapan.F0142',
    'female_25_29':         '5YearIncrementsEsriJapan.F0143',
    'female_30_34':         '5YearIncrementsEsriJapan.F0144',
}
ENRICH_VARS = list(ENRICH_VAR_MAP.values())

# Whitespace optimization: only enrich variables the deployed model actually uses.
# Model needs (via 04 derive_features): pop_total, pop_female_total, households_total,
# household_income_avg, beverages_total, beverages_tea, beverages_tea_drinks,
# female_10_14, female_15_19, female_20_24. 17 -> 10 variables (~40% savings).
if MODEL_FEATURES_ONLY_FOR_WHITESPACE and REQUIRED_DEPLOYMENT_FEATURES is not None:
    _whitespace_keep_keys = {
        'pop_total', 'pop_female_total', 'households_total', 'household_income_avg',
        'beverages_total', 'beverages_tea', 'beverages_tea_drinks',
        'female_10_14', 'female_15_19', 'female_20_24',
    }
    ENRICH_VAR_MAP = {k: v for k, v in ENRICH_VAR_MAP.items() if k in _whitespace_keep_keys}
    ENRICH_VARS = list(ENRICH_VAR_MAP.values())
    print(f'[whitespace mode] pruned ENRICH_VAR_MAP -> {len(ENRICH_VAR_MAP)} model-required vars')

# Inline validation: confirm every ID exists in the catalog and print the alias (year).
_cat_path = OUT_TABLES / 'esri_japan_data_collections.csv'
if _cat_path.exists():
    _cat = pd.read_csv(_cat_path)
    _lookup = dict(zip(_cat['analysisVariable'].astype(str), _cat['alias'].astype(str)))
    print(f'Variables to enrich: {len(ENRICH_VARS)}\n')
    missing = []
    for k, v in ENRICH_VAR_MAP.items():
        alias = _lookup.get(v)
        if alias is None:
            print(f'  {k:25s} <- {v:50s} *** NOT FOUND ***')
            missing.append(k)
        else:
            print(f'  {k:25s} <- {v:50s} | {alias}')
    if missing:
        print(f'\nWARNING: {len(missing)} variable(s) not in catalog: {missing}')
else:
    print('Catalog CSV not found. Run cell 3c with DISCOVER_VARS=True first.')
    print(f'Variables to enrich: {len(ENRICH_VARS)}')
    for k, v in ENRICH_VAR_MAP.items():
        print(f'  {k:25s} <- {v}')

[whitespace mode] pruned ENRICH_VAR_MAP -> 10 model-required vars
Variables to enrich: 10

  pop_total                 <- PopulationTotalsEsriJapan.F0103                    | 2025 Total Population by Age
  pop_female_total          <- PopulationTotalsEsriJapan.F0137                    | 2025 Female Population by Age
  households_total          <- HouseholdsbyIncomeEsriJapan.I_BASE                 | 2025 Total Households by Income
  household_income_avg      <- HouseholdsbyIncomeEsriJapan.I_AVE_GENHHHO          | 2025 Average Household Income
  beverages_total           <- FoodEsriJapan.C0214                                | 2026 Beverages
  beverages_tea             <- FoodEsriJapan.C0215                                | 2026 Beverages/Tea
  beverages_tea_drinks      <- FoodEsriJapan.C0219                                | 2026 Beverages/Tea: Tea Beverages
  female_10_14              <- 5YearIncrementsEsriJapan.F0140                     | 2025 Female Population Age 10-14
  female_15_19 

In [12]:
# 3e-DRYRUN. Before committing all credits, run enrich on a tiny sample to measure burn rate.
# Set DRYRUN_N = 0 to skip; default samples 50 anchors and reports actual credits consumed
# per anchor plus a budget-vs-needed projection. Self-contained: doesn't depend on cell 3e.
DRYRUN_N = 50

if RUN_ESRI_ENRICH and POP_SOURCE == 'esri' and study_areas is not None and DRYRUN_N > 0:
    # Avoid double-charging: skip anchors already on disk in the checkpoint dir.
    _partial_dir = ROOT / 'output' / 'cache' / 'esri_enrich_partial' / ANCHOR_MODE
    _done = set()
    if _partial_dir.exists():
        for f in _partial_dir.glob('*.parquet'):
            try:
                _done.update(pd.read_parquet(f, columns=['anchor_id'])['anchor_id'].astype(str).tolist())
            except Exception:
                pass
    sample = study_areas[~study_areas['anchor_id'].astype(str).isin(_done)].head(DRYRUN_N).copy()

    if len(sample) == 0:
        print('Dry run skipped: first', DRYRUN_N, 'anchors already enriched in checkpoint dir.')
    else:
        credits_before = float(gis_private.admin.credits.credits)
        print(f'Credits BEFORE dry run : {credits_before:>14,.2f}')
        from arcgis.geoenrichment import enrich
        ok = 0
        try:
            dr = enrich(study_areas=sample, analysis_variables=ENRICH_VARS,
                        gis=gis_private, return_geometry=False)
            ok = len(dr)
        except Exception as e:
            print(f'Dry run failed: {e}')
        credits_after = float(gis_private.admin.credits.credits)
        used = credits_before - credits_after
        print(f'Credits AFTER dry run  : {credits_after:>14,.2f}')
        print(f'Credits used           : {used:>14,.4f} on {ok} anchors x {len(ENRICH_VARS)} vars')
        if ok > 0 and used > 0:
            per_anchor = used / ok
            per_attr = used / (ok * len(ENRICH_VARS))
            full_n = len(study_areas)
            projected = per_anchor * full_n
            print()
            print(f'  per anchor             : {per_anchor:>12,.4f} credits')
            print(f'  per attribute (var)    : {per_attr:>12,.4f} credits')
            print(f'  full run ({full_n:,} anchors) projection:')
            print(f'    credits needed       : {projected:>14,.0f}')
            print(f'    credits available    : {credits_after:>14,.0f}')
            headroom_pct = (credits_after - projected) / credits_after * 100
            print(f'    budget headroom      : {headroom_pct:>13.1f}%')
            if projected > credits_after:
                print('  WARNING: full run will exceed available credits — bump WHITESPACE_MIN_MESH_POP')
            else:
                print('  OK: full run fits within credit budget.')
        else:
            print('Could not measure burn rate (dry run returned 0 rows or 0 credits used).')
else:
    print('Dry run skipped.')

Credits BEFORE dry run :     983,402.44
Credits AFTER dry run  :     983,402.44
Credits used           :         0.0000 on 50 anchors x 10 vars
Could not measure burn rate (dry run returned 0 rows or 0 credits used).


In [13]:
# 3e. Batch enrich with BISECT fallback + per-batch checkpointing + permanent-error skip.
#
# Robustness model:
#  - Every successful batch is saved IMMEDIATELY to disk as a parquet shard.
#  - Re-running this cell auto-resumes: anchors already on disk are skipped.
#  - PERMANENT failures (no ESRI address coverage) are recorded in `_failed.csv`
#    so they're skipped on resume too — no infinite retry on rivers / forests.
#  - Kernel crashes / Ctrl+C / power loss only cost the in-flight batch.
#  - Partial shards live in: output/cache/esri_enrich_partial/{ANCHOR_MODE}/
#  - To force a fresh re-enrich: delete that folder.
#
# Error taxonomy:
#  - permanent : "Error Code: 400" / "Unable to find address" — coordinate has
#                no ESRI address data (rivers, lakes, large forests, certain
#                industrial / coastal zones). Retrying does nothing; we drop it.
#  - transient : "default_dataset" — known arcgis SDK bug where lazy-loaded
#                country attributes go stale under load. Retry once after a
#                short sleep usually clears it.
#  - unknown   : anything else (timeouts, network) — retry once just in case.
#
# Bisect fallback:
#  - With per-anchor fallback, a 50-batch with 2 bad anchors costs ~50 API calls.
#  - Bisect isolates the bad anchors in ~2 * log2(N) calls per failure,
#    cutting fallback time 4-10x for typical bad-anchor density.
import time
from collections import Counter

ESRI_PARTIAL_DIR = ROOT / 'output' / 'cache' / 'esri_enrich_partial' / ANCHOR_MODE
ESRI_PARTIAL_DIR.mkdir(parents=True, exist_ok=True)
ESRI_FAILED_LOG = ESRI_PARTIAL_DIR / '_failed.csv'


def _shard_path(anchor_ids):
    """File name for the parquet shard covering a list/array of anchor ids."""
    ids = [str(x) for x in anchor_ids]
    first, last = ids[0], ids[-1]
    return ESRI_PARTIAL_DIR / f'shard_{first}__{last}.parquet'


def _classify_exception(exc):
    s = str(exc)
    if 'Error Code: 400' in s or 'Unable to find address' in s:
        return 'permanent'
    if 'default_dataset' in s:
        return 'transient'
    return 'unknown'


def _enrich_once(subset, analysis_variables, gis):
    """Single attempt at enrichment — no retry."""
    from arcgis.geoenrichment import enrich
    try:
        out = enrich(study_areas=subset, analysis_variables=analysis_variables,
                     gis=gis, return_geometry=False)
        out = out.reset_index(drop=True)
        out['anchor_id'] = subset['anchor_id'].values[:len(out)]
        return out, None
    except Exception as e:
        return None, e


def _enrich_with_retry(subset, analysis_variables, gis, transient_sleep_s=3):
    """One attempt + a single retry only for transient/unknown errors.
    Permanent errors are returned immediately so we don't waste time."""
    out, exc = _enrich_once(subset, analysis_variables, gis)
    if out is not None:
        return out, None
    if _classify_exception(exc) in ('transient', 'unknown'):
        time.sleep(transient_sleep_s)
        out, exc = _enrich_once(subset, analysis_variables, gis)
        if out is not None:
            return out, None
    return None, exc


def _bisect_enrich(subset, analysis_variables, gis):
    """Recursively bisect a failing batch to isolate the bad anchor(s).

    Returns (saved_count, failed_records).
    Each successful sub-batch is written to disk immediately.
    """
    out, exc = _enrich_with_retry(subset, analysis_variables, gis)
    if out is not None:
        out.to_parquet(_shard_path(subset['anchor_id'].tolist()), index=False)
        return len(out), []

    if len(subset) == 1:
        aid = str(subset['anchor_id'].iloc[0])
        return 0, [(aid, _classify_exception(exc), str(exc)[:160])]

    mid = len(subset) // 2
    left_n, left_failed = _bisect_enrich(subset.iloc[:mid], analysis_variables, gis)
    right_n, right_failed = _bisect_enrich(subset.iloc[mid:], analysis_variables, gis)
    return left_n + right_n, left_failed + right_failed


def _already_done_anchor_ids():
    """Scan partial parquet shards to find anchors already enriched (resume state)."""
    done = set()
    for f in ESRI_PARTIAL_DIR.glob('*.parquet'):
        try:
            sh = pd.read_parquet(f, columns=['anchor_id'])
            done.update(sh['anchor_id'].astype(str).tolist())
        except Exception as e:
            print(f'  WARN: could not read {f.name} ({e}); deleting and re-enriching its anchors')
            f.unlink(missing_ok=True)
    return done


def _already_failed_anchor_ids():
    """Persistent permanent-failure log — these anchors are skipped on resume."""
    if not ESRI_FAILED_LOG.exists():
        return set()
    try:
        df = pd.read_csv(ESRI_FAILED_LOG)
        # Only treat PERMANENT failures as 'skip' on resume; transient/unknown get another shot
        mask = df['error_class'] == 'permanent'
        return set(df.loc[mask, 'anchor_id'].astype(str).tolist())
    except Exception:
        return set()


def _append_failures(failed_records):
    """Append (anchor_id, error_class, error_msg) rows to the failure log."""
    if not failed_records:
        return
    new = pd.DataFrame(failed_records, columns=['anchor_id', 'error_class', 'error_msg'])
    if ESRI_FAILED_LOG.exists():
        new.to_csv(ESRI_FAILED_LOG, mode='a', index=False, header=False, encoding='utf-8-sig')
    else:
        new.to_csv(ESRI_FAILED_LOG, index=False, encoding='utf-8-sig')


def enrich_loop_checkpointed(study_areas, analysis_variables, gis, batch_size=50):
    """Resumable enrichment with bisect fallback.

    Returns (enriched_df, still_failed_anchor_ids).
    """
    from tqdm.auto import tqdm

    all_ids = study_areas['anchor_id'].astype(str).tolist()
    done = _already_done_anchor_ids()
    perm_failed = _already_failed_anchor_ids()
    skip = done | perm_failed
    pending_mask = ~study_areas['anchor_id'].astype(str).isin(skip)
    pending = study_areas.loc[pending_mask].reset_index(drop=True)
    print(f'Resume status   : {len(done):>8,} done | {len(perm_failed):>6,} prior-permanent | '
          f'{len(pending):>8,} pending | {len(all_ids):>8,} total')
    print(f'Checkpoint dir  : {ESRI_PARTIAL_DIR}')
    print(f'Failure log     : {ESRI_FAILED_LOG.name}')

    new_failed = []
    n_saved_total = 0
    n_perm_total = 0
    n_trans_total = 0

    if len(pending):
        pbar = tqdm(range(0, len(pending), batch_size), desc='enrich pass 1')
        for start in pbar:
            subset = pending.iloc[start:start + batch_size]
            out, exc = _enrich_with_retry(subset, analysis_variables, gis)
            if out is not None:
                out.to_parquet(_shard_path(subset['anchor_id'].tolist()), index=False)
                n_saved_total += len(out)
                continue
            # Batch failed → bisect to isolate the bad anchor(s)
            saved_n, failed_rec = _bisect_enrich(subset, analysis_variables, gis)
            n_saved_total += saved_n
            if failed_rec:
                _append_failures(failed_rec)
                new_failed.extend(failed_rec)
                cls_counts = Counter(f[1] for f in failed_rec)
                n_perm_total += cls_counts.get('permanent', 0)
                n_trans_total += cls_counts.get('transient', 0) + cls_counts.get('unknown', 0)
                pbar.set_postfix(saved=n_saved_total,
                                 perm=n_perm_total, trans_fail=n_trans_total)

    # Concat all shards on disk
    parts = []
    for f in sorted(ESRI_PARTIAL_DIR.glob('*.parquet')):
        parts.append(pd.read_parquet(f))
    df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

    if len(df):
        df = df[df['anchor_id'].astype(str).isin(all_ids)].reset_index(drop=True)

    still_failed_ids = [f[0] for f in new_failed]
    return df, still_failed_ids


if RUN_ESRI_ENRICH and POP_SOURCE == 'esri' and study_areas is not None:
    df_enriched_long, still_failed_ids = enrich_loop_checkpointed(
        study_areas, ENRICH_VARS, gis_private, batch_size=ESRI_ENRICH_BATCH_SIZE)
    print(f'\nEnriched rows: {len(df_enriched_long):,} / {len(study_areas):,} | '
          f'unrecoverable this run: {len(still_failed_ids)}')
    df_enriched_long.to_csv(OUT_TABLES / 'esri_enriched_long.csv', index=False, encoding='utf-8-sig')
else:
    df_enriched_long = pd.DataFrame()
    print('Skipping ESRI enrich loop')

Resume status   :        0 done |      0 prior-permanent |    2,786 pending |    2,786 total
Checkpoint dir  : c:\Users\52333\OneDrive - Bain\Desktop\CSE\T5HM\output\cache\esri_enrich_partial\whitespace_sc
Failure log     : _failed.csv


enrich pass 1: 100%|██████████| 56/56 [05:44<00:00,  6.15s/it]



Enriched rows: 2,786 / 2,786 | unrecoverable this run: 0


In [14]:
df_enriched_long

,source_country,aggregation_method,population_to_polygon_size_rating,apportionment_confidence,has_data,f0140,f0141,f0142,c0214,c0215,c0219,i_base,i_ave_genhhho,f0103,f0137,anchor_id
0,JP,BlockApportionment:JP.Blocks;PointsLayer:JP.Bl...,-1,-1,1,3,3,3,4463106,868269,586228,81.9,4587912,166,77,wssc_10204004
1,JP,BlockApportionment:JP.Blocks;PointsLayer:JP.Bl...,-1,-1,1,9,9,7,5159805,996408,663744,89.9,4845384,251,128,wssc_10204006
2,JP,BlockApportionment:JP.Blocks;PointsLayer:JP.Bl...,-1,-1,1,1,1,1,852809,164107,108609,14.2,4658451,32,18,wssc_10207002
3,JP,BlockApportionment:JP.Blocks;PointsLayer:JP.Bl...,-1,-1,1,8,8,8,9700343,1872161,1245805,179.5,4480501,363,178,wssc_10209001
4,JP,BlockApportionment:JP.Blocks;PointsLayer:JP.Bl...,-1,-1,1,0,0,0,0,0,0,0.0,0,0,0,wssc_10210002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2781,JP,BlockApportionment:JP.Blocks;PointsLayer:JP.Bl...,-1,-1,1,3,3,4,6683115,1303698,884521,100.3,4744766,186,84,wssc_10202007
2782,JP,BlockApportionment:JP.Blocks;PointsLayer:JP.Bl...,-1,-1,1,3,3,3,3538341,682670,453994,53.3,5133208,126,66,wssc_10202008
2783,JP,BlockApportionment:JP.Blocks;PointsLayer:JP.Bl...,-1,-1,1,19,18,18,24028554,4673950,3154908,379.4,4936874,818,439,wssc_10202009
2784,JP,BlockApportionment:JP.Blocks;PointsLayer:JP.Bl...,-1,-1,1,7,7,4,10693701,2082647,1408878,198.6,4323263,365,202,wssc_10203002


In [15]:
# 3f. Reshape enriched output to one row per anchor + derive comparison-friendly metrics.
# Single catchment now -> no pivot over radii, just rename ESRI columns to business names.
def pivot_enriched_to_anchor_features(df_long, var_map, side_m=CATCHMENT_SIDE_M):
    if df_long.empty:
        return pd.DataFrame()
    col_lookup = {c.lower(): c for c in df_long.columns}
    out = df_long[['anchor_id']].drop_duplicates('anchor_id').copy()
    suffix = f'_{int(side_m)}m'  # e.g. _500m
    for biz_name, esri_id in var_map.items():
        tail = esri_id.split('.')[-1].lower()
        src_col = col_lookup.get(tail)
        if src_col is None:
            print(f'  (skip) {biz_name} -> {esri_id} not in enriched output')
            continue
        out = out.merge(
            df_long[['anchor_id', src_col]].rename(columns={src_col: f'{biz_name}{suffix}'}),
            on='anchor_id', how='left')
    # Derive women_10_29 if all 4 female age bands are present
    female_bands = ['female_10_14', 'female_15_19', 'female_20_24', 'female_25_29']
    fcols = [f'{b}{suffix}' for b in female_bands if f'{b}{suffix}' in out.columns]
    if len(fcols) == len(female_bands):
        out[f'women_10_29{suffix}'] = out[fcols].sum(axis=1)
    out = out.drop_duplicates('anchor_id').reset_index(drop=True)

    # --- Derived comparison-friendly metrics ---
    # Catchment area in km^2 (square of side_m on each side)
    area_km2 = (side_m / 1000.0) ** 2  # 500m square -> 0.25 km^2
    def safe_div(a, b):
        return np.where((b > 0) & b.notna(), a / b, np.nan) if hasattr(b, 'notna') else (a / b if b else np.nan)
    def col(name):
        return out[name] if name in out.columns else None

    hh   = col(f'households_total{suffix}')
    pop  = col(f'pop_total{suffix}')
    bev  = col(f'beverages_total{suffix}')
    tea  = col(f'beverages_tea{suffix}')
    tea_d = col(f'beverages_tea_drinks{suffix}')
    w1029 = col(f'women_10_29{suffix}')
    f_pop  = col(f'pop_female_total{suffix}')

    # Per-household spending (most useful unit for comparing anchors)
    if bev is not None and hh is not None:
        out[f'beverages_per_household{suffix}'] = safe_div(bev, hh)
    if tea is not None and hh is not None:
        out[f'beverages_tea_per_household{suffix}'] = safe_div(tea, hh)
    if tea_d is not None and hh is not None:
        out[f'beverages_tea_drinks_per_household{suffix}'] = safe_div(tea_d, hh)

    # Share of tea spending within total beverages (qualitative preference for tea)
    if tea is not None and bev is not None:
        out[f'tea_share_of_beverages{suffix}'] = safe_div(tea, bev)
    if tea_d is not None and bev is not None:
        out[f'tea_drinks_share_of_beverages{suffix}'] = safe_div(tea_d, bev)

    # Population density (people per km^2) and target-customer density (women 10-29 / km^2)
    if pop is not None:
        out[f'pop_density_per_km2{suffix}'] = pop / area_km2
    if w1029 is not None:
        out[f'women_10_29_density_per_km2{suffix}'] = w1029 / area_km2

    # Target-customer share of population
    if w1029 is not None and pop is not None:
        out[f'women_10_29_share_of_pop{suffix}'] = safe_div(w1029, pop)
    if f_pop is not None and pop is not None:
        out[f'female_share_of_pop{suffix}'] = safe_div(f_pop, pop)

    return out

if not df_enriched_long.empty:
    anchor_demographics = pivot_enriched_to_anchor_features(df_enriched_long, ENRICH_VAR_MAP, CATCHMENT_SIDE_M)
    csv_write(anchor_demographics, OUT_TABLES / 'anchor_demographics_esri.csv')
else:
    anchor_demographics = pd.DataFrame()
    print('No enriched data yet')

Wrote c:\Users\52333\OneDrive - Bain\Desktop\CSE\T5HM\output\tables\anchor_demographics_esri.csv | 2786 rows, 18 cols


In [16]:
anchor_demographics

,anchor_id,pop_total_500m,pop_female_total_500m,households_total_500m,household_income_avg_500m,beverages_total_500m,beverages_tea_500m,beverages_tea_drinks_500m,female_10_14_500m,female_15_19_500m,female_20_24_500m,beverages_per_household_500m,beverages_tea_per_household_500m,beverages_tea_drinks_per_household_500m,tea_share_of_beverages_500m,tea_drinks_share_of_beverages_500m,pop_density_per_km2_500m,female_share_of_pop_500m
0,wssc_10204004,166,77,81.9,4587912,4463106,868269,586228,3,3,3,54494.578755,10601.575092,7157.851038,0.194544,0.131350,664.0,0.463855
1,wssc_10204006,251,128,89.9,4845384,5159805,996408,663744,9,9,7,57394.938821,11083.515017,7383.136819,0.193110,0.128637,1004.0,0.509960
2,wssc_10207002,32,18,14.2,4658451,852809,164107,108609,1,1,1,60056.971831,11556.830986,7648.521127,0.192431,0.127354,128.0,0.562500
3,wssc_10209001,363,178,179.5,4480501,9700343,1872161,1245805,8,8,8,54040.908078,10429.866295,6940.417827,0.192999,0.128429,1452.0,0.490358
4,wssc_10210002,0,0,0.0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2781,wssc_10202007,186,84,100.3,4744766,6683115,1303698,884521,3,3,4,66631.256231,12997.986042,8818.753739,0.195073,0.132352,744.0,0.451613
2782,wssc_10202008,126,66,53.3,5133208,3538341,682670,453994,3,3,3,66385.384615,12808.067542,8517.711069,0.192935,0.128307,504.0,0.523810
2783,wssc_10202009,818,439,379.4,4936874,24028554,4673950,3154908,19,18,18,63333.036373,12319.319979,8315.519241,0.194516,0.131298,3272.0,0.536675
2784,wssc_10203002,365,202,198.6,4323263,10693701,2082647,1408878,7,7,4,53845.422961,10486.641490,7094.048338,0.194755,0.131748,1460.0,0.553425


## 4. Competitors

Standardize each brand into the POI schema: `poi_id, poi_source, poi_type, poi_brand, name, address, lat, lng`.
These tables become the right-hand side of the Alteryx pairwise distance workflow.

In [17]:
# 4a. Genuine = Gongcha itself (target brand being DD-ed).
# Reuse the cleaned Gongcha table as the Genuine POI set so it can be counted in catchments.
# poi_id uses the same 'gc_' prefix as anchor_id so that exclude_self=True in
# count_pois_in_square() correctly drops the anchor's own store from its own catchment.
# is_inside_sc (IS/RS from VDR) is attached so Section 8a-3 can split into
# inside-SC / outside-SC subsets for the attribute-list 'Presence/# of Genuine' breakdown.
poi_genuine = pd.DataFrame({
    'poi_id': 'gc_' + gc_open['gongcha_id'].astype(str),
    'poi_source': 'gongcha_internal',
    'poi_type': 'genuine',
    'poi_brand': 'Gongcha',
    'name': gc_open['gongcha_name'],
    'address': gc_open['address'],
    'lat': gc_open['lat'],
    'lng': gc_open['lng'],
}).dropna(subset=['lat', 'lng']).reset_index(drop=True)

# Attach is_inside_sc from VDR (same xlsx as anchor's is_inside_sc; covers ALL 217 open stores,
# whereas anchors only has the 186 mature ones).
_gc_xlsx = DATA_DIR / 'internal' / '260513_global_store_address_vshare_ジオコーディング修正_手作業追加_vup.xlsx'
_isrs = pd.read_excel(_gc_xlsx, sheet_name='Sheet2_Database', header=9, usecols=['No.', 'IS/RS']).dropna(subset=['No.'])
_isrs['poi_id'] = 'gc_' + pd.to_numeric(_isrs['No.'], errors='coerce').astype('Int64').astype(str)
_isrs['is_inside_sc'] = (_isrs['IS/RS'].astype(str).str.strip().str.upper() == 'IS').astype('Int8')
poi_genuine = poi_genuine.merge(_isrs[['poi_id', 'is_inside_sc']], on='poi_id', how='left')

_n_in  = int((poi_genuine['is_inside_sc'] == 1).sum())
_n_out = int((poi_genuine['is_inside_sc'] == 0).sum())
_n_na  = int(poi_genuine['is_inside_sc'].isna().sum())
print(f'Genuine POI rows : {len(poi_genuine)}')
print(f'  inside SC      : {_n_in}')
print(f'  outside SC     : {_n_out}')
print(f'  unknown IS/RS  : {_n_na}')

Genuine POI rows : 217
  inside SC      : 206
  outside SC     : 11
  unknown IS/RS  : 0


In [18]:
# 4b. Starbucks — reuse the cleaned adhoc table
sb = pd.read_csv(OUT_TABLES / 'adhoc_starbucks_points.csv')
poi_starbucks = pd.DataFrame({
    'poi_id': 'sb_' + sb['starbucks_id'].astype(str),
    'poi_source': 'starbucks_v2',
    'poi_type': 'competitor_coffee',
    'poi_brand': 'Starbucks',
    'name': sb['store_name'],
    'address': sb['address'],
    'lat': sb['lat'],
    'lng': sb['lng'],
})
poi_starbucks = poi_starbucks.dropna(subset=['lat', 'lng']).reset_index(drop=True)
print('Starbucks POI rows:', len(poi_starbucks))

Starbucks POI rows: 2122


In [19]:
# 4c. Tully's — already has lat/lng in dataBank file
tul = pd.read_excel(DATA_DIR / 'competitors' / '260515_dataBank_tullys_JP.xlsx')
tul.columns = [str(c).strip() for c in tul.columns]
lat_c = next((c for c in tul.columns if c.lower() in ('latitude', 'lat')), None)
lng_c = next((c for c in tul.columns if c.lower() in ('longitude', 'lng', 'lon')), None)
name_c = next((c for c in tul.columns if c.lower() in ('name', 'storename', 'store_name')), None)
addr_c = next((c for c in tul.columns if c.lower() in ('fulladdress', 'address')), None)
id_c = next((c for c in tul.columns if c.lower() == 'storeid'), None)
poi_tullys = pd.DataFrame({
    'poi_id': 'tul_' + tul[id_c].astype(str) if id_c else 'tul_' + tul.index.astype(str),
    'poi_source': 'tullys_databank',
    'poi_type': 'competitor_coffee',
    'poi_brand': "Tully's",
    'name': tul[name_c] if name_c else pd.NA,
    'address': tul[addr_c] if addr_c else pd.NA,
    'lat': to_numeric_safe(tul[lat_c]) if lat_c else np.nan,
    'lng': to_numeric_safe(tul[lng_c]) if lng_c else np.nan,
})
poi_tullys = poi_tullys.dropna(subset=['lat', 'lng']).reset_index(drop=True)
print("Tully's POI rows:", len(poi_tullys))

Tully's POI rows: 843


In [20]:
# 4d. Other tea brands (CHA BAR, Pearl Lady, 春水堂, Bull Pulu, The Alley)
# Source file has address only. Geocoding step is pending — Yahoo Japan API or Google geocoding.
tb = pd.read_excel(DATA_DIR / 'competitors' / 'Tea brand_store_addresses.xlsx')
poi_tea_brands_raw = pd.DataFrame({
    'poi_id': 'tb_' + tb.index.astype(str),
    'poi_source': 'tea_brand_search_list',
    'poi_type': 'competitor_tea',
    'poi_brand': tb['Company'].astype(str) + '|' + tb['ブランド'].astype(str),
    'name': tb['店舗名'],
    'address': tb['住所'],
    'lat': np.nan,
    'lng': np.nan,
})
print('Tea-brand raw rows (need geocoding):', len(poi_tea_brands_raw))
print('Brand mix:')
print(poi_tea_brands_raw['poi_brand'].value_counts())
# When geocoded, fill lat/lng then concat with poi_starbucks / poi_tullys / poi_genuine.

Tea-brand raw rows (need geocoding): 99
Brand mix:
poi_brand
Bull Pulu|Bull Pulu      37
Pearl Lady|Pearl Lady    20
Pearl Lady|CHA BAR       18
春水堂|春水堂                  13
THE ALLEY|THE ALLEY      11
Name: count, dtype: int64


In [21]:
# 4e. Geocode tea-brand addresses using ESRI geocoding service.
# Cached: first run hits ESRI; subsequent runs read from poi_tea_brands_geocoded.csv (no credits).
# Force re-geocode by deleting that CSV.
#
# Two-pass strategy:
#   pass 1 - raw address as-is
#   pass 2 - for any row with score < GEOCODE_MIN_SCORE, strip trailing building-name /
#            floor / parenthetical noise and re-geocode. Most JP geocoder misses come
#            from "...123-4 BuildingName 1F" tails confusing the parser.
import re

TEA_BRANDS_GEO_CACHE = OUT_TABLES / 'poi_tea_brands_geocoded.csv'
GEOCODE_MIN_SCORE = 80  # ESRI returns 0-100 match score; 80+ is typically reliable

def clean_jp_address(addr):
    """Strip parenthetical notes and trailing building/floor info from a JP address.
    Keeps everything up to the last "数字-数字" / "丁目数字番" / "数字番(地)" pattern."""
    s = str(addr).strip()
    s = re.sub(r'[（(][^)）]*[)）]', '', s)  # drop (...) and （...）
    # Find the END of the last house-number-like token. Patterns covered:
    #   "1-2-3", "1981-3" (hyphen chains; supports - ー －)
    #   "2丁目16番3号", "2丁目16番", "2丁目16"
    #   "947番地", "947番"
    pat = (r'[0-9０-９]+(?:[-ー－][0-9０-９]+)+'
           r'|[0-9０-９]+丁目[0-9０-９]+(?:番(?:地)?[0-9０-９]*(?:号)?)?'
           r'|[0-9０-９]+番(?:地)?[0-9０-９]*(?:号)?')
    matches = list(re.finditer(pat, s))
    if matches:
        s = s[:matches[-1].end()]
    return s.strip()

def esri_geocode_one(addr):
    """Single-address geocode. Returns dict(lat, lng, score, match_addr).
    arcgis 2.x uses the active GIS automatically (no `gis` kwarg)."""
    from arcgis.geocoding import geocode
    try:
        results = geocode(address=str(addr), source_country='JPN', max_locations=1,
                          as_featureset=False)
        if not results:
            return {'lat': np.nan, 'lng': np.nan, 'score': np.nan, 'match_addr': None}
        r = results[0]
        loc = r.get('location', {})
        attrs = r.get('attributes', {})
        return {
            'lat': loc.get('y', np.nan),
            'lng': loc.get('x', np.nan),
            'score': attrs.get('Score', np.nan),
            'match_addr': attrs.get('Match_addr'),
        }
    except Exception as e:
        return {'lat': np.nan, 'lng': np.nan, 'score': np.nan, 'match_addr': f'ERROR: {e}'}

RUN_TEA_BRANDS_GEOCODE = True  # flip False to skip section entirely
if RUN_TEA_BRANDS_GEOCODE and TEA_BRANDS_GEO_CACHE.exists():
    geo = pd.read_csv(TEA_BRANDS_GEO_CACHE)
    print(f'Loaded cached geocoding result: {len(geo)} rows from {TEA_BRANDS_GEO_CACHE.name}')

    # Pass 2: retry any low-score rows with a cleaned address (no API hit if already 80+)
    if gis_private is not None and (geo['score'].fillna(0) < GEOCODE_MIN_SCORE).any():
        from tqdm.auto import tqdm
        low_mask = geo['score'].fillna(0) < GEOCODE_MIN_SCORE
        low_ids = geo.loc[low_mask, 'poi_id'].tolist()
        addr_lookup = dict(zip(poi_tea_brands_raw['poi_id'], poi_tea_brands_raw['address']))
        print(f'\nRetrying {len(low_ids)} low-score rows with cleaned addresses...')
        n_improved = 0
        for pid in tqdm(low_ids, desc='ESRI geocode pass 2'):
            raw_addr = addr_lookup.get(pid)
            cleaned = clean_jp_address(raw_addr)
            if cleaned == raw_addr:
                continue  # nothing to clean
            res = esri_geocode_one(cleaned)
            old_score = geo.loc[geo['poi_id'] == pid, 'score'].iloc[0]
            new_score = res.get('score') or 0
            if new_score > (old_score or 0):
                for k in ('lat', 'lng', 'score', 'match_addr'):
                    geo.loc[geo['poi_id'] == pid, k] = res[k]
                n_improved += 1
        if n_improved:
            csv_write(geo, TEA_BRANDS_GEO_CACHE)
            print(f'Pass 2 improved {n_improved} rows; cache updated.')
        else:
            print('Pass 2 did not improve any rows.')
elif RUN_TEA_BRANDS_GEOCODE and gis_private is not None:
    from tqdm.auto import tqdm
    rows = []
    for _, row in tqdm(poi_tea_brands_raw.iterrows(), total=len(poi_tea_brands_raw),
                       desc='ESRI geocode pass 1'):
        res = esri_geocode_one(row['address'])
        res['poi_id'] = row['poi_id']
        res['used_cleaned_addr'] = False
        rows.append(res)
    geo = pd.DataFrame(rows)

    # Pass 2: retry low-score rows with cleaned address
    low_mask = geo['score'].fillna(0) < GEOCODE_MIN_SCORE
    if low_mask.any():
        addr_lookup = dict(zip(poi_tea_brands_raw['poi_id'], poi_tea_brands_raw['address']))
        print(f'\nPass 2: retrying {low_mask.sum()} low-score rows with cleaned addresses')
        for pid in tqdm(geo.loc[low_mask, 'poi_id'].tolist(), desc='ESRI geocode pass 2'):
            cleaned = clean_jp_address(addr_lookup[pid])
            if cleaned == addr_lookup[pid]:
                continue
            res = esri_geocode_one(cleaned)
            old_score = geo.loc[geo['poi_id'] == pid, 'score'].iloc[0]
            new_score = res.get('score') or 0
            if new_score > (old_score or 0):
                for k in ('lat', 'lng', 'score', 'match_addr'):
                    geo.loc[geo['poi_id'] == pid, k] = res[k]
                geo.loc[geo['poi_id'] == pid, 'used_cleaned_addr'] = True
    csv_write(geo, TEA_BRANDS_GEO_CACHE)
else:
    geo = pd.DataFrame(columns=['poi_id', 'lat', 'lng', 'score', 'match_addr'])
    print('Skipping geocoding (RUN_TEA_BRANDS_GEOCODE=False or no ESRI session)')

# Merge geocoded lat/lng/score onto the raw table and drop the placeholder NaN coords
poi_tea_brands = poi_tea_brands_raw.drop(columns=['lat', 'lng']).merge(
    geo[['poi_id', 'lat', 'lng', 'score', 'match_addr']], on='poi_id', how='left')

# Quality cuts: drop nulls and low-confidence matches
n_total = len(poi_tea_brands)
poi_tea_brands_all = poi_tea_brands.copy()  # keep full table for QA
poi_tea_brands = poi_tea_brands.dropna(subset=['lat', 'lng']).copy()
n_with_coord = len(poi_tea_brands)
poi_tea_brands = poi_tea_brands[poi_tea_brands['score'].fillna(0) >= GEOCODE_MIN_SCORE].reset_index(drop=True)
n_high_conf = len(poi_tea_brands)

print(f'\nGeocoding summary:')
print(f'  total addresses     : {n_total}')
print(f'  got coordinates     : {n_with_coord}')
print(f'  score >= {GEOCODE_MIN_SCORE}        : {n_high_conf}')
if n_high_conf < n_total:
    print(f'  *** {n_total - n_high_conf} addresses dropped; review poi_tea_brands_all for diagnostics ***')
print()
print('Per-brand pass rate:')
print(poi_tea_brands_all.assign(passed=poi_tea_brands_all['score'].fillna(0) >= GEOCODE_MIN_SCORE)
      .groupby('poi_brand')['passed'].agg(['sum', 'count']))

Loaded cached geocoding result: 99 rows from poi_tea_brands_geocoded.csv

Geocoding summary:
  total addresses     : 99
  got coordinates     : 99
  score >= 80        : 99

Per-brand pass rate:
                       sum  count
poi_brand                        
Bull Pulu|Bull Pulu     37     37
Pearl Lady|CHA BAR      18     18
Pearl Lady|Pearl Lady   20     20
THE ALLEY|THE ALLEY     11     11
春水堂|春水堂                 13     13


In [22]:
# 4f. Mister Donut — already has lat/lng (no geocoding needed).
# Modeled as a competitor on the "casual sweet/snack indulgence" axis, complementary to
# bubble tea but in the same daypart (afternoon visits in malls + transit hubs).
md = pd.read_excel(DATA_DIR / 'competitors' / '260520_T5HM_misterDonut_JP.xlsx')
md = md.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

poi_mister_donut = pd.DataFrame({
    'poi_id':     'md_' + md.index.astype(str),
    'poi_source': 'mister_donut_v1',
    'poi_type':   'competitor_food',
    'poi_brand':  'Mister Donut',
    'name':       md['name'],
    'address':    md['fullAddress'],
    'lat':        md['latitude'],
    'lng':        md['longitude'],
})
print(f'Mister Donut POI rows: {len(poi_mister_donut)}')
print(f'  lat range: {poi_mister_donut["lat"].min():.2f} – {poi_mister_donut["lat"].max():.2f}')
print(f'  lng range: {poi_mister_donut["lng"].min():.2f} – {poi_mister_donut["lng"].max():.2f}')
poi_mister_donut.head(3)


Mister Donut POI rows: 1073
  lat range: 24.34 – 45.40
  lng range: 124.16 – 144.98


,poi_id,poi_source,poi_type,poi_brand,name,address,lat,lng
0,md_0,mister_donut_v1,competitor_food,Mister Donut,イオン札幌桑園 ショップ,北海道札幌市中央区北８条西１４丁目２８番,43.069576,141.333559
1,md_1,mister_donut_v1,competitor_food,Mister Donut,大通駅 ショップ,北海道札幌市中央区大通西４丁目地下鉄南北線大通駅構内,43.059890,141.352091
2,md_2,mister_donut_v1,competitor_food,Mister Donut,すすきの ショップ,北海道札幌市中央区南四条西３丁目グリーンビル,43.055512,141.354613


In [23]:
# 4g. Combine competitor POIs (only those with lat/lng) into a single table
_parts = [poi_starbucks, poi_tullys]
if 'poi_tea_brands' in dir() and len(poi_tea_brands):
    # Keep only the standard POI columns when concatenating
    _tb = poi_tea_brands[[c for c in poi_starbucks.columns if c in poi_tea_brands.columns]].copy()
    _parts.append(_tb)
if 'poi_mister_donut' in dir() and len(poi_mister_donut):
    _md = poi_mister_donut[[c for c in poi_starbucks.columns if c in poi_mister_donut.columns]].copy()
    _parts.append(_md)
poi_competitors = pd.concat(_parts, ignore_index=True)
print('Combined competitor POIs with coords:', len(poi_competitors))
print(poi_competitors.groupby('poi_brand').size().to_string())

Combined competitor POIs with coords: 4137
poi_brand
Bull Pulu|Bull Pulu        37
Mister Donut             1073
Pearl Lady|CHA BAR         18
Pearl Lady|Pearl Lady      20
Starbucks                2122
THE ALLEY|THE ALLEY        11
Tully's                   843
春水堂|春水堂                    13


In [24]:
poi_competitors.head()

,poi_id,poi_source,poi_type,poi_brand,name,address,lat,lng
0,sb_1522,starbucks_v2,competitor_coffee,Starbucks,古川店,989-6117 宮城県 大崎市 古川旭2-3-15,38.566068,140.972148
1,sb_4460,starbucks_v2,competitor_coffee,Starbucks,古川北稲葉店,989-6145 宮城県 大崎市 古川北稲葉3-4-12,38.564664,140.948415
2,sb_1466,starbucks_v2,competitor_coffee,Starbucks,秋田駅店,010-0001 秋田県 秋田市 中通7-1-2,39.716674,140.129179
3,sb_1236,starbucks_v2,competitor_coffee,Starbucks,秋田東通店,010-0003 秋田県 秋田市 東通4-5-20,39.712896,140.143214
4,sb_1516,starbucks_v2,competitor_coffee,Starbucks,秋田保戸野学園通り店,010-0913 秋田県 秋田市 保戸野鉄砲町14-18,39.724801,140.108860


## 5. Shopping centers

SC list_v1 sheet 1 has 3,093 SCs with X/Y, 店舗面積 (sales floor area), 売上高 (sales). The header row is row 7 (0-indexed 6).

In [25]:
sc_xls = pd.ExcelFile(DATA_DIR / 'shopping_center' / '260517_SC list_v1.xlsx')
sc_db_sheet = next((s for s in sc_xls.sheet_names if 'SC' in s and ('情報' in s or '売上' in s)), sc_xls.sheet_names[1])
sc_raw = pd.read_excel(sc_xls, sheet_name=sc_db_sheet, header=6)
sc_raw.columns = [str(c).strip() for c in sc_raw.columns]
print('SC sheet:', sc_db_sheet, '| shape:', sc_raw.shape)
print('Cols:', list(sc_raw.columns)[:20])

SC sheet: SC情報（売上データ付き） | shape: (3093, 27)
Cols: ['Unnamed: 0', 'RecID', 'INDEX', 'X', 'Y', 'SC名称', 'SC名称読み', '郵便番号', '都道府県名', '市区町村名', '所在地', '所在地ビル名', '駐車台数', '敷地面積', '建物延床面積', '店舗面積', 'テナント面積', 'テナント数総計', '順位', '売上高（百万円）']


In [26]:
# Standardize SC columns. 'X' = longitude, 'Y' = latitude in this file
sc_col_map = {
    'RecID': 'sc_id', 'SC名称': 'name', '都道府県名': 'prefecture_jp', '市区町村名': 'city_jp',
    '所在地': 'address', '店舗面積': 'sales_floor_m2', '建物延床面積': 'building_floor_m2',
    'テナント数総数': 'tenant_count', '売上高（百万円）': 'sales_mn_jpy', '営業面積(㎡)': 'op_floor_m2',
    'X': 'lng', 'Y': 'lat',
}
sc = sc_raw.rename(columns=sc_col_map)
keep_cols = [v for v in sc_col_map.values() if v in sc.columns]
sc = sc[keep_cols].copy()
for c in ['lat', 'lng', 'sales_floor_m2', 'building_floor_m2', 'tenant_count', 'sales_mn_jpy', 'op_floor_m2']:
    if c in sc.columns:
        sc[c] = to_numeric_safe(sc[c])
sc = sc.dropna(subset=['lat', 'lng']).reset_index(drop=True)

poi_sc = pd.DataFrame({
    'poi_id': 'sc_' + sc['sc_id'].astype(str),
    'poi_source': 'sc_list_v1',
    'poi_type': 'shopping_center',
    'poi_brand': pd.NA,
    'name': sc['name'],
    'address': sc['address'],
    'lat': sc['lat'],
    'lng': sc['lng'],
    'sales_floor_m2': sc['sales_floor_m2'],
    'sales_mn_jpy': sc.get('sales_mn_jpy'),
})
print('SC POI rows:', len(poi_sc))
poi_sc.head(3)

SC POI rows: 3093


,poi_id,poi_source,poi_type,poi_brand,name,address,lat,lng,sales_floor_m2,sales_mn_jpy
0,sc_1101004,sc_list_v1,shopping_center,<NA>,さっぽろ地下街オーロラタウン・ポールタウン,南一条西4,43.058168,141.352668,13488.0,12094.0
1,sc_1101007,sc_list_v1,shopping_center,<NA>,札幌ＰＡＲＣＯ（札幌パルコ）,南一条西3-3,43.058761,141.353157,14000.0,16655.0
2,sc_1101012,sc_list_v1,shopping_center,<NA>,サッポロファクトリー,北二条東4丁目1-2,43.064941,141.362678,44631.7,14003.0


## 6. Schools

MLIT P29-23 学校 shapefile covers all school types in Japan (~50K points). All are kept and tagged with a broad bucket (`school_type`). Catchment-relevant slices (`university`, `high_school`, `vocational`) are counted separately in Section 8a-4 on top of an overall total. The category-code column position varies between MLIT vintages, so we detect it by value range (16400-16499) rather than hard-coding `P29_004`.

In [27]:
def load_schools_mlit():
    """Extract MLIT P29-23 school shapefile, return ALL schools tagged with a broad
    catchment-relevant bucket. P29_xxx field positions vary between vintages, so we
    detect the 学校分類コード column by content (values in 16400-16499 range)."""
    import zipfile, tempfile
    import geopandas as gpd
    zpath = DATA_DIR / 'school' / 'school-P29-23_GML.zip'
    with tempfile.TemporaryDirectory() as td:
        with zipfile.ZipFile(zpath) as z:
            z.extractall(td)
        shp = next(Path(td).rglob('*.shp'))
        gdf = gpd.read_file(shp)

    # Find the school-category code column by content: integer values in 16000-16499.
    # In P29-23 it's P29_003 (codes are 16001-16016); P29_004 holds the school name.
    category_col = None
    for c in [col for col in gdf.columns if col.startswith('P29')]:
        s = pd.to_numeric(gdf[c], errors='coerce')
        if s.between(16000, 16499).mean() > 0.9:
            category_col = c
            break
    if category_col is None:
        raise RuntimeError(f'Could not find school category column. P29 cols: '
                           f'{[c for c in gdf.columns if c.startswith("P29")]}')
    print(f'Detected school category column: {category_col}')

    # MLIT P29-23 学校分類 codes (160xx series) -> broad catchment bucket.
    # Keep ALL 13 school types; unknown codes (future MLIT versions) fall into 'other'.
    code_map = {
        16001: 'elementary',     # 小学校
        16002: 'middle_school',  # 中学校
        16003: 'high_school',    # 中等教育学校 (6-yr combined secondary)
        16004: 'high_school',    # 高等学校
        16005: 'vocational',     # 高等専門学校 (5-yr tech college)
        16006: 'university',     # 短期大学 (junior college, same target demographic)
        16007: 'university',     # 大学
        16011: 'kindergarten',   # 幼稚園
        16012: 'special_needs',  # 特別支援学校
        16013: 'kindergarten',   # 認定こども園
        16014: 'elementary',     # 義務教育学校 (1-9 yr integrated, majority elementary-age)
        16015: 'vocational',     # 各種学校
        16016: 'vocational',     # 専修学校
    }
    codes_int = pd.to_numeric(gdf[category_col], errors='coerce').astype('Int64')
    gdf['school_type'] = codes_int.map(code_map).fillna('other')
    gdf = gdf.to_crs(epsg=4326)
    name_col = next((c for c in gdf.columns if '名' in c or 'NAME' in c.upper()), None)
    addr_col = next((c for c in gdf.columns if '所在' in c or 'ADDRESS' in c.upper()), None)
    return pd.DataFrame({
        'poi_id': 'sch_' + gdf.index.astype(str),
        'poi_source': 'mlit_p29_2023',
        'poi_type': 'school_' + gdf['school_type'].astype(str),
        'poi_brand': pd.NA,
        'name': gdf[name_col] if name_col else pd.NA,
        'address': gdf[addr_col] if addr_col else pd.NA,
        'lat': gdf.geometry.y,
        'lng': gdf.geometry.x,
        'school_type': gdf['school_type'],
    })

LOAD_SCHOOLS = True  # flip off to skip schools entirely
if LOAD_SCHOOLS:
    poi_schools = load_schools_mlit()
    print('School POI rows:', len(poi_schools))
    print()
    print('Breakdown by school_type:')
    print(poi_schools['school_type'].value_counts())
else:
    poi_schools = pd.DataFrame(columns=STD_POI_COLS + ['school_type'])
    print('Skipping schools (flag off)')

Detected school category column: P29_003
School POI rows: 56807

Breakdown by school_type:
school_type
elementary       19189
kindergarten     15822
middle_school     9946
high_school       5000
vocational        4096
university        1525
special_needs     1229
Name: count, dtype: int64


## 7. Region tag, density tier, stations & Shinkansen flag

For each anchor:
- spatial join with ADM1 (prefecture) and ADM2 (municipality) — drives Region cut in the model
- assign density tier from a prefecture lookup — kept as urban/rural cohort feature
- load MLIT **S12-25 駅別乗降客数** as `poi_stations` (10.5K stations, 14-yr passenger time series 2011–2024) — station foot traffic POI
- subset to Shinkansen lines as `poi_shinkansen` (113 stations) — both catchment-level and municipality-level proximity flags

In [28]:
# Prefecture density tier — kept as a feature (urban/rural cohort).
# NOTE: catchment is now a single fixed square (CATCHMENT_SIDE_M), so this no longer drives catchment size.
PREFECTURE_TIER = {
    # high-density urban
    'Tokyo': 'high_density', 'Osaka': 'high_density', 'Kanagawa': 'high_density',
    'Aichi': 'high_density',
    # medium
    'Saitama': 'medium_density', 'Chiba': 'medium_density', 'Hyogo': 'medium_density',
    'Kyoto': 'medium_density', 'Fukuoka': 'medium_density', 'Hokkaido': 'medium_density',
    'Miyagi': 'medium_density', 'Hiroshima': 'medium_density',
}

In [29]:
def attach_admin_tags(anchors_df):
    """Spatial join anchors → ADM1 (prefecture) and ADM2 (municipality). Derive density_tier.
    Idempotent: safe to re-run on an anchors table that already has the admin cols."""
    import geopandas as gpd
    # Drop any pre-existing output cols so re-runs don't create duplicates
    _drop = ['prefecture_en', 'prefecture_jp', 'municipality_en', 'municipality_jp', 'density_tier']
    anchors_df = anchors_df.drop(columns=[c for c in _drop if c in anchors_df.columns])
    adm1 = gpd.read_file(DATA_DIR / 'admin_shp' / 'jpn_adm_2019_shp' / 'jpn_admbnda_adm1_2019.shp')[['ADM1_EN', 'ADM1_JA', 'geometry']]
    adm2 = gpd.read_file(DATA_DIR / 'admin_shp' / 'jpn_adm_2019_shp' / 'jpn_admbnda_adm2_2019.shp')[['ADM2_EN', 'ADM2_JA', 'ADM1_EN', 'geometry']]
    gdf = gpd.GeoDataFrame(
        anchors_df.copy(),
        geometry=gpd.points_from_xy(anchors_df['lng'], anchors_df['lat']),
        crs='EPSG:4326')
    gdf = gpd.sjoin(gdf, adm1, how='left', predicate='within').drop(columns='index_right')
    gdf = gdf.rename(columns={'ADM1_EN': 'prefecture_en', 'ADM1_JA': 'prefecture_jp'})
    gdf = gpd.sjoin(gdf, adm2[['ADM2_EN', 'ADM2_JA', 'geometry']], how='left', predicate='within').drop(columns='index_right')
    gdf = gdf.rename(columns={'ADM2_EN': 'municipality_en', 'ADM2_JA': 'municipality_jp'})
    df = pd.DataFrame(gdf.drop(columns='geometry'))
    df['density_tier'] = df['prefecture_en'].map(PREFECTURE_TIER).fillna('low_density')
    return df

RUN_ADMIN_JOIN = True  # needed for shinkansen_served_municipality flag (see next cell)
if RUN_ADMIN_JOIN:
    anchors = attach_admin_tags(anchors)
    print('Anchors with region tags:', anchors.shape)
    print(anchors['density_tier'].value_counts())
else:
    print('Skipping admin spatial join (flag off)')

Anchors with region tags: (2786, 16)
density_tier
low_density    2786
Name: count, dtype: int64


In [30]:
# Load MLIT S12-25 駅別乗降客数 (Number of Passengers per Station).
# This single dataset gives us BOTH:
#   1. Station POI table with annual passenger counts (foot-traffic proxy)
#   2. Shinkansen subset for flag features (catchment-level + municipality-level)
#
# S12 schema (verified against the actual file):
#   S12_001 = 駅名 (station name)
#   S12_002 = 運営会社 (operator)
#   S12_003 = 路線名  (line name; rows containing "新幹線" are Shinkansen)
#   S12_004 = 鉄道区分コード (11=JR, 12=私鉄, 13=JR新幹線, ...)
#   S12_009/013/.../061 = annual passenger counts, 4 cols per year, 2011 -> 2024
#                          We expose 2024 (latest, S12_061) and 2019 (pre-COVID, S12_041).
# Geometry is LineString (platform / track segment); we centroid it to a Point for spatial use.

def load_stations_s12():
    """Return (poi_stations, poi_shinkansen) standardized POI tables."""
    import geopandas as gpd
    shp = DATA_DIR / 'shikansen' / 'S12-25_GML' / 'UTF-8' / 'S12-25_NumberOfPassengers.shp'
    gdf = gpd.read_file(shp, encoding='utf-8').to_crs(epsg=4326)
    # LineString -> representative Point (centroid is fine for short station segments)
    gdf['lat'] = gdf.geometry.centroid.y
    gdf['lng'] = gdf.geometry.centroid.x

    # Annual passenger columns (verified empirically: every 4th col from S12_009).
    YEAR_COLS = {
        2019: 'S12_041',   # pre-COVID baseline
        2024: 'S12_061',   # latest available
    }
    for yr, col in YEAR_COLS.items():
        gdf[f'passengers_{yr}'] = pd.to_numeric(gdf[col], errors='coerce').fillna(0)

    df = pd.DataFrame({
        'poi_id': 'stn_' + gdf.index.astype(str),
        'poi_source': 'mlit_s12_2025',
        'poi_type': 'station',
        'poi_brand': gdf['S12_002'],   # operator
        'name': gdf['S12_001'],         # station name
        'line_name': gdf['S12_003'],
        'rail_type_code': gdf['S12_004'],
        'address': pd.NA,
        'lat': gdf['lat'],
        'lng': gdf['lng'],
        'passengers_2019': gdf['passengers_2019'],
        'passengers_2024': gdf['passengers_2024'],
    }).dropna(subset=['lat', 'lng']).reset_index(drop=True)

    # Some stations show up multiple times (one row per line/segment they belong to).
    # IMPORTANT: dedup Shinkansen subset SEPARATELY from the all-stations table.
    # If we dedup first then filter "line_name contains 新幹線", a station that has
    # both a Shinkansen line and a regular line gets collapsed onto the higher-traffic
    # regular line and disappears from the Shinkansen subset.
    shink_raw = df[df['line_name'].astype(str).str.contains('新幹線', na=False)].copy()

    def _dedupe(d):
        return (d.sort_values(['poi_brand', 'name', 'passengers_2024'],
                              ascending=[True, True, False])
                 .drop_duplicates(subset=['poi_brand', 'name'], keep='first')
                 .reset_index(drop=True))

    df = _dedupe(df)
    shink = _dedupe(shink_raw)
    return df, shink

LOAD_STATIONS = True  # flip off to skip
if LOAD_STATIONS:
    poi_stations, poi_shinkansen = load_stations_s12()
    print(f'Station POI rows         : {len(poi_stations)}')
    print(f'  of which Shinkansen    : {len(poi_shinkansen)}')
    print(f'  passengers_2024 total  : {poi_stations["passengers_2024"].sum():,.0f}')
    print()
    print('Top 5 stations by passengers_2024:')
    print(poi_stations.nlargest(5, 'passengers_2024')[['name', 'poi_brand', 'line_name', 'passengers_2024']]
          .to_string(index=False))
    print()
    print('Shinkansen line breakdown:')
    print(poi_shinkansen['line_name'].value_counts())
else:
    poi_stations = pd.DataFrame(columns=STD_POI_COLS + ['passengers_2019', 'passengers_2024'])
    poi_shinkansen = poi_stations.copy()
    print('Skipping stations (flag off)')

# Municipality-level Shinkansen flag — set on the anchor table directly.
# Requires admin tags (Section 7 cell above with RUN_ADMIN_JOIN=True).
if len(poi_shinkansen) and 'municipality_jp' in anchors.columns:
    import geopandas as gpd
    adm2 = gpd.read_file(DATA_DIR / 'admin_shp' / 'jpn_adm_2019_shp' / 'jpn_admbnda_adm2_2019.shp')[['ADM2_JA', 'geometry']]
    shink_pts = gpd.GeoDataFrame(
        poi_shinkansen[['poi_id', 'lat', 'lng']],
        geometry=gpd.points_from_xy(poi_shinkansen['lng'], poi_shinkansen['lat']),
        crs='EPSG:4326')
    served = gpd.sjoin(shink_pts, adm2, how='left', predicate='within')['ADM2_JA'].dropna().unique()
    anchors['shinkansen_served_municipality'] = anchors['municipality_jp'].isin(set(served)).astype(int)
    print(f'\nMunicipalities served by Shinkansen: {len(served)}')
    print(f'Anchors in a Shinkansen-served municipality: {anchors["shinkansen_served_municipality"].sum()} / {len(anchors)}')
else:
    if 'shinkansen_served_municipality' not in anchors.columns:
        anchors['shinkansen_served_municipality'] = pd.NA
    print('\nShinkansen municipality flag skipped (no poi_shinkansen or no municipality column)')

Station POI rows         : 9774
  of which Shinkansen    : 108
  passengers_2024 total  : 119,551,975

Top 5 stations by passengers_2024:
name poi_brand line_name  passengers_2024
  渋谷      東急電鉄       東横線          1770430
  新宿   東日本旅客鉄道       中央線          1333618
  池袋   東日本旅客鉄道       山手線           998256
  東京   東日本旅客鉄道      東海道線           869128
  大阪   西日本旅客鉄道      東海道線           751006

Shinkansen line breakdown:
line_name
東北新幹線     22
北陸新幹線     19
山陽新幹線     19
東海道新幹線    17
九州新幹線     12
上越新幹線     10
西九州新幹線     5
北海道新幹線     4
Name: count, dtype: int64

Municipalities served by Shinkansen: 102
Anchors in a Shinkansen-served municipality: 690 / 2786


## 8. Assemble final feature table

This is where **all the counting happens**. Sections 4–7 only *build* the standardized POI tables. This section is the single consumer that runs the same `build_block()` pipeline on each of them.

Four kinds of features per POI source:

1. **In-catchment count** — # of POIs falling inside each anchor's `CATCHMENT_SIDE_M × CATCHMENT_SIDE_M` square. Computed via `geopandas.sjoin` (primary feature aligned with the methodology).
2. **Wider context rings** (1 / 3 / 5 km) — # of POIs within each ring distance. Computed via vectorized haversine.
3. **(optional) Nearest POI** — id + distance of closest POI; used for Gongcha-self (cannibalization), stations and Shinkansen.
4. **(optional) Value sums** — sum of numeric POI attributes (e.g. SC sales, station passengers) over square + each ring; reuses the masks from (1) and (2).

**Execution order (designed so you can stop / resume mid-pipeline):**

- **8a** — build all non-foot-traffic blocks (fast, ~5 sec)
- **8b** — save `model_features.csv`  ⭐ **first checkpoint**
- **8c** — ingest MLIT 1km mesh foot traffic (heavy, ~5-10 min first run, parquet-cached after)
- **8d** — build foot traffic block + merge + save `model_features_full.csv`  ⭐ **final checkpoint**

If 8c hangs or you ^C, your `model_features.csv` from 8b is intact and complete except for foot traffic.

In [31]:
# 8a. Build feature blocks — everything in Python, no Alteryx dependency.
# Each POI source gets:
#   - In-catchment count  : # POIs inside the 500m x 500m square (geopandas sjoin)
#   - Ring counts         : # POIs within 1 / 3 / 5 km (vectorized haversine)
#   - (optional) Nearest  : id + distance of closest POI (cannibalization signal)
#   - (optional) Value sums: sum of numeric POI attributes (e.g. SC sales/floor area)
#                            inside square + each ring

def pairwise_haversine_km(lats_a, lngs_a, lats_b, lngs_b):
    """Great-circle distance in km between two point sets. Returns shape (len(a), len(b))."""
    R = 6371.0088  # WGS84 mean radius
    la = np.radians(np.asarray(lats_a, dtype=float))
    lo = np.radians(np.asarray(lngs_a, dtype=float))
    lb = np.radians(np.asarray(lats_b, dtype=float))
    lob = np.radians(np.asarray(lngs_b, dtype=float))
    dlat = lb[None, :] - la[:, None]
    dlng = lob[None, :] - lo[:, None]
    a = np.sin(dlat / 2) ** 2 + np.cos(la[:, None]) * np.cos(lb[None, :]) * np.sin(dlng / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def count_pois_in_square(anchor_df, poi_df, side_m=CATCHMENT_SIDE_M, exclude_self=False):
    """# POIs inside each anchor's CATCHMENT_SIDE_M square (axis-aligned in EPSG:3857)."""
    import geopandas as gpd
    from shapely.geometry import box
    half = side_m / 2
    a = anchor_df[['anchor_id', 'lat', 'lng']].dropna(subset=['lat', 'lng']).copy()
    a_pts = gpd.GeoDataFrame(a, geometry=gpd.points_from_xy(a['lng'], a['lat']),
                             crs='EPSG:4326').to_crs('EPSG:3857')
    a_pts['geometry'] = a_pts.geometry.apply(
        lambda pt: box(pt.x - half, pt.y - half, pt.x + half, pt.y + half))
    p = poi_df[['poi_id', 'lat', 'lng']].dropna(subset=['lat', 'lng']).copy()
    if len(p) == 0:
        return anchor_df[['anchor_id']].drop_duplicates().assign(cnt=0)
    p_pts = gpd.GeoDataFrame(p, geometry=gpd.points_from_xy(p['lng'], p['lat']),
                             crs='EPSG:4326').to_crs('EPSG:3857')
    joined = gpd.sjoin(p_pts, a_pts[['anchor_id', 'geometry']], how='inner', predicate='within')
    if exclude_self:
        joined = joined[joined['poi_id'].astype(str) != joined['anchor_id'].astype(str)]
    cnt = joined.groupby('anchor_id').size().reset_index(name='cnt')
    return anchor_df[['anchor_id']].drop_duplicates().merge(cnt, on='anchor_id', how='left').fillna({'cnt': 0})

def count_pois_in_rings(anchor_df, poi_df, rings_km, prefix, exclude_self=False):
    """# POIs within each ring (km) per anchor via vectorized haversine.
    exclude_self=True masks pairs where poi_id == anchor_id (Gongcha self-count)."""
    a = anchor_df[['anchor_id', 'lat', 'lng']].dropna(subset=['lat', 'lng']).reset_index(drop=True)
    out = a[['anchor_id']].copy()
    if poi_df is None or len(poi_df) == 0:
        for km in rings_km:
            out[f'{prefix}_cnt_{int(km)}km'] = 0
        return out
    p = poi_df[['poi_id', 'lat', 'lng']].dropna(subset=['lat', 'lng']).reset_index(drop=True)
    D = pairwise_haversine_km(a['lat'].values, a['lng'].values, p['lat'].values, p['lng'].values)
    if exclude_self:
        # .to_numpy() is required: pandas 2.x string dtype -> ArrowStringArray, which
        # does NOT support [:, None] 2D indexing. .values returns the extension array.
        a_ids = a['anchor_id'].astype(str).to_numpy()
        p_ids = p['poi_id'].astype(str).to_numpy()
        eq = (a_ids[:, None] == p_ids[None, :])
        D = np.where(eq, np.inf, D)
    for km in rings_km:
        out[f'{prefix}_cnt_{int(km)}km'] = (D <= km).sum(axis=1).astype(int)
    return out

def nearest_poi(anchor_df, poi_df, prefix, exclude_self=False):
    """Nearest POI per anchor: id + distance in km."""
    a = anchor_df[['anchor_id', 'lat', 'lng']].dropna(subset=['lat', 'lng']).reset_index(drop=True)
    out = a[['anchor_id']].copy()
    if poi_df is None or len(poi_df) == 0:
        out[f'nearest_{prefix}_id'] = pd.NA
        out[f'nearest_{prefix}_dist_km'] = np.nan
        return out
    p = poi_df[['poi_id', 'lat', 'lng']].dropna(subset=['lat', 'lng']).reset_index(drop=True)
    D = pairwise_haversine_km(a['lat'].values, a['lng'].values, p['lat'].values, p['lng'].values)
    if exclude_self:
        a_ids = a['anchor_id'].astype(str).to_numpy()
        p_ids = p['poi_id'].astype(str).to_numpy()
        eq = (a_ids[:, None] == p_ids[None, :])
        D = np.where(eq, np.inf, D)
    idx = D.argmin(axis=1)
    out[f'nearest_{prefix}_id'] = p['poi_id'].values[idx]
    out[f'nearest_{prefix}_dist_km'] = D[np.arange(len(a)), idx]
    return out

def sum_pois_in_square(anchor_df, poi_df, value_cols, side_m=CATCHMENT_SIDE_M, exclude_self=False):
    """Sum each value_col over POIs inside each anchor's CATCHMENT_SIDE_M square.
    Columns missing in poi_df are returned as 0. Output cols: {value_col}_sum."""
    import geopandas as gpd
    from shapely.geometry import box
    half = side_m / 2
    present = [c for c in value_cols if c in poi_df.columns]
    a = anchor_df[['anchor_id', 'lat', 'lng']].dropna(subset=['lat', 'lng']).copy()
    a_pts = gpd.GeoDataFrame(a, geometry=gpd.points_from_xy(a['lng'], a['lat']),
                             crs='EPSG:4326').to_crs('EPSG:3857')
    a_pts['geometry'] = a_pts.geometry.apply(
        lambda pt: box(pt.x - half, pt.y - half, pt.x + half, pt.y + half))
    p_cols = ['poi_id', 'lat', 'lng'] + present
    p = poi_df[p_cols].dropna(subset=['lat', 'lng']).copy()
    for c in present:
        p[c] = pd.to_numeric(p[c], errors='coerce').fillna(0.0)
    out = anchor_df[['anchor_id']].drop_duplicates().copy()
    if len(p) == 0:
        for vc in value_cols:
            out[f'{vc}_sum'] = 0.0
        return out
    p_pts = gpd.GeoDataFrame(p, geometry=gpd.points_from_xy(p['lng'], p['lat']),
                             crs='EPSG:4326').to_crs('EPSG:3857')
    joined = gpd.sjoin(p_pts, a_pts[['anchor_id', 'geometry']], how='inner', predicate='within')
    if exclude_self:
        joined = joined[joined['poi_id'].astype(str) != joined['anchor_id'].astype(str)]
    for vc in value_cols:
        if vc not in present:
            out[f'{vc}_sum'] = 0.0; continue
        s = joined.groupby('anchor_id')[vc].sum().rename(f'{vc}_sum').reset_index()
        out = out.merge(s, on='anchor_id', how='left')
        out[f'{vc}_sum'] = out[f'{vc}_sum'].fillna(0.0)
    return out

def sum_pois_in_rings(anchor_df, poi_df, value_cols, rings_km, prefix, exclude_self=False):
    """Sum each value_col over POIs within each ring (km) per anchor.
    Output cols: {prefix}_{value_col}_sum_{km}km."""
    a = anchor_df[['anchor_id', 'lat', 'lng']].dropna(subset=['lat', 'lng']).reset_index(drop=True)
    out = a[['anchor_id']].copy()
    present = [c for c in value_cols if (poi_df is not None and c in poi_df.columns)]
    if poi_df is None or len(poi_df) == 0:
        for km in rings_km:
            for vc in value_cols:
                out[f'{prefix}_{vc}_sum_{int(km)}km'] = 0.0
        return out
    p_cols = ['poi_id', 'lat', 'lng'] + present
    p = poi_df[p_cols].dropna(subset=['lat', 'lng']).reset_index(drop=True)
    D = pairwise_haversine_km(a['lat'].values, a['lng'].values, p['lat'].values, p['lng'].values)
    if exclude_self:
        a_ids = a['anchor_id'].astype(str).to_numpy()
        p_ids = p['poi_id'].astype(str).to_numpy()
        eq = (a_ids[:, None] == p_ids[None, :])
        D = np.where(eq, np.inf, D)
    for km in rings_km:
        mask = (D <= km).astype(float)
        for vc in value_cols:
            col = f'{prefix}_{vc}_sum_{int(km)}km'
            if vc not in present:
                out[col] = 0.0; continue
            vals = pd.to_numeric(p[vc], errors='coerce').fillna(0.0).values
            out[col] = mask @ vals
    return out

side_tag = f'{int(CATCHMENT_SIDE_M)}m_sq'  # e.g. '500m_sq'
RINGS_KM = [1, 3, 5]
feature_blocks = {}

def _need_direct_feature(col):
    """Whether a direct 03 output column is required for whitespace scoring."""
    if not MODEL_FEATURES_ONLY_FOR_WHITESPACE or REQUIRED_DEPLOYMENT_FEATURES is None:
        return True
    return col in REQUIRED_DEPLOYMENT_FEATURES


def build_block(name, poi_df, exclude_self=False, with_nearest=False, value_cols=None):
    """Build POI features.

    Existing-store training mode keeps the full diagnostic feature set.
    Whitespace modes, when the saved 04 model bundle is available, keep only direct
    columns required by the deployed model. This avoids expensive 1/3/5km rings and
    nearest-POI features for whole-Japan scoring.
    """
    if poi_df is None or len(poi_df) == 0:
        print(f'  (skip) {name:20s}: empty POI table')
        return None

    blk = anchors[['anchor_id']].drop_duplicates().copy()

    # 500m-square count
    sq_col = f'{name}_in_{side_tag}_cnt'
    if _need_direct_feature(sq_col):
        sq = count_pois_in_square(anchors, poi_df, exclude_self=exclude_self).rename(columns={'cnt': sq_col})
        blk = blk.merge(sq, on='anchor_id', how='left')

    # 1/3/5km rings (training diagnostics only unless the deployed model requires them)
    ring_cols = [f'{name}_cnt_{int(km)}km' for km in RINGS_KM]
    needed_ring_cols = [c for c in ring_cols if _need_direct_feature(c)]
    if needed_ring_cols:
        rings = count_pois_in_rings(anchors, poi_df, RINGS_KM, name, exclude_self=exclude_self)
        blk = blk.merge(rings[['anchor_id'] + needed_ring_cols], on='anchor_id', how='left')

    # Nearest POI features
    nearest_cols = [f'nearest_{name}_id', f'nearest_{name}_dist_km']
    needed_nearest_cols = [c for c in nearest_cols if _need_direct_feature(c)]
    if with_nearest and needed_nearest_cols:
        near = nearest_poi(anchors, poi_df, name, exclude_self=exclude_self)
        blk = blk.merge(near[['anchor_id'] + needed_nearest_cols], on='anchor_id', how='left')

    # Numeric POI attribute sums
    if value_cols:
        square_sum_cols = {vc: f'{name}_{vc}_sum_in_{side_tag}' for vc in value_cols}
        needed_square_value_cols = [vc for vc, out_col in square_sum_cols.items() if _need_direct_feature(out_col)]
        if needed_square_value_cols:
            sq_sum = sum_pois_in_square(anchors, poi_df, needed_square_value_cols, exclude_self=exclude_self)
            sq_sum = sq_sum.rename(columns={f'{vc}_sum': square_sum_cols[vc] for vc in needed_square_value_cols})
            blk = blk.merge(sq_sum, on='anchor_id', how='left')

        needed_ring_value_cols = []
        for vc in value_cols:
            for km in RINGS_KM:
                out_col = f'{name}_{vc}_sum_{int(km)}km'
                if _need_direct_feature(out_col):
                    needed_ring_value_cols.append(vc)
                    break
        if needed_ring_value_cols:
            ring_sum = sum_pois_in_rings(anchors, poi_df, needed_ring_value_cols, RINGS_KM, name,
                                         exclude_self=exclude_self)
            keep_cols = ['anchor_id'] + [c for c in ring_sum.columns if _need_direct_feature(c)]
            blk = blk.merge(ring_sum[keep_cols], on='anchor_id', how='left')

    print(f'  {name:20s}: {blk.shape[0]:>3} anchors x {blk.shape[1]-1:>2} feature cols')
    return blk

# 8a-1. Per-brand competitor blocks
for name, df in [
    ('starbucks',    poi_starbucks),
    ('tullys',       poi_tullys),
    ('tea_brands',   poi_tea_brands if 'poi_tea_brands' in dir() else None),
    ('mister_donut', poi_mister_donut if 'poi_mister_donut' in dir() else None),
]:
    blk = build_block(name, df)
    if blk is not None:
        feature_blocks[name] = blk

# 8a-2. Combined competitor count (Starbucks + Tully's + tea brands)
if 'poi_competitors' in dir() and len(poi_competitors):
    blk = build_block('competitors', poi_competitors)
    if blk is not None:
        feature_blocks['competitors'] = blk

# 8a-3. Gongcha self (cannibalization) — exclude_self drops the anchor's own store.
# Three variables per attribute list: total, inside-SC subset, outside-SC subset.
# 'inside_sc' / 'outside_sc' use the VDR IS/RS flag attached in 4a.
gc_blk = build_block('gongcha', poi_genuine, exclude_self=True, with_nearest=True)
if gc_blk is not None:
    feature_blocks['gongcha'] = gc_blk

if 'is_inside_sc' in poi_genuine.columns:
    poi_genuine_inside  = poi_genuine[poi_genuine['is_inside_sc'] == 1]
    poi_genuine_outside = poi_genuine[poi_genuine['is_inside_sc'] == 0]
    blk = build_block('gongcha_inside_sc',  poi_genuine_inside,  exclude_self=True)
    if blk is not None:
        feature_blocks['gongcha_inside_sc'] = blk
    blk = build_block('gongcha_outside_sc', poi_genuine_outside, exclude_self=True)
    if blk is not None:
        feature_blocks['gongcha_outside_sc'] = blk

# 8a-4. Shopping centers + schools
# SC: size-weighted sums of sales floor area and sales (a 100k m^2 mall is a very
# different catchment driver than a 1k m^2 strip).
sc_value_cols = [c for c in ['sales_floor_m2', 'sales_mn_jpy'] if c in poi_sc.columns]
blk = build_block('sc', poi_sc, value_cols=sc_value_cols)
if blk is not None:
    feature_blocks['sc'] = blk

# 8a-5. Stations + Shinkansen (foot-traffic + rail connectivity)
# Stations get passenger-weighted sums -> direct foot-traffic proxy per catchment.
# Shinkansen gets count-only -> binary-ish "is there a Shinkansen station within X km" feature.
if 'poi_stations' in dir() and len(poi_stations):
    station_value_cols = [c for c in ['passengers_2019', 'passengers_2024'] if c in poi_stations.columns]
    blk = build_block('stations', poi_stations, with_nearest=True, value_cols=station_value_cols)
    if blk is not None:
        feature_blocks['stations'] = blk
if 'poi_shinkansen' in dir() and len(poi_shinkansen):
    blk = build_block('shinkansen', poi_shinkansen, with_nearest=True)
    if blk is not None:
        feature_blocks['shinkansen'] = blk

# Schools: one overall block + 3 sliced blocks for the catchment-relevant categories.
# Target customer = 10-29 yo women, so each slice maps to a distinct demand pattern:
#   - university (incl. 短期大学): 18-22 yo on-campus, strong fit for premium tea
#   - high_school (incl. 中等教育学校): 15-18 yo after-school traffic, lower ticket
#   - vocational (高等専門 / 専修 / 各種学校): 18-22 yo skill-track, similar to univ.
# kindergarten / elementary / middle_school / special_needs stay in the overall total
# only — not in their own slice (not the buyer cohort).
SCHOOL_SLICES = ['university', 'high_school', 'vocational']
if len(poi_schools):
    blk = build_block('schools', poi_schools)
    if blk is not None:
        feature_blocks['schools'] = blk
    for st in SCHOOL_SLICES:
        sub = poi_schools[poi_schools['school_type'] == st]
        blk = build_block(f'schools_{st}', sub if len(sub) else None)
        if blk is not None:
            feature_blocks[f'schools_{st}'] = blk
else:
    print(f'  (skip) {"schools":20s}: empty POI table')

# 8a-6. ESRI demographics (from Section 3f)
if not anchor_demographics.empty:
    feature_blocks['esri_demographics'] = anchor_demographics
    print(f'  {"esri_demographics":20s}: {anchor_demographics.shape[0]:>3} anchors x {anchor_demographics.shape[1]-1:>2} feature cols')
else:
    print('  esri_demographics   : empty (run Section 3 first)')

  starbucks           : 2786 anchors x  1 feature cols
  tullys              : 2786 anchors x  1 feature cols
  tea_brands          : 2786 anchors x  1 feature cols
  mister_donut        : 2786 anchors x  1 feature cols
  competitors         : 2786 anchors x  1 feature cols
  gongcha             : 2786 anchors x  1 feature cols
  gongcha_inside_sc   : 2786 anchors x  1 feature cols
  gongcha_outside_sc  : 2786 anchors x  0 feature cols
  sc                  : 2786 anchors x  3 feature cols
  stations            : 2786 anchors x  2 feature cols
  shinkansen          : 2786 anchors x  1 feature cols
  schools             : 2786 anchors x  1 feature cols
  schools_university  : 2786 anchors x  1 feature cols
  schools_high_school : 2786 anchors x  0 feature cols
  schools_vocational  : 2786 anchors x  1 feature cols
  esri_demographics   : 2786 anchors x 17 feature cols


In [32]:
# 8b. Join all feature blocks onto anchors -> mode-specific model_features*.csv
features = anchors.copy()
for src, block in feature_blocks.items():
    features = features.merge(block, on='anchor_id', how='left')

# Fill missing _cnt_ cols with 0 (no nearby POIs of that type)
cnt_cols = [c for c in features.columns if '_cnt_' in c]
for c in cnt_cols:
    features[c] = features[c].fillna(0).astype(int)

csv_write(features, FEATURE_PATH,
          sanitize_cols=[c for c in ['anchor_name','address'] if c in features.columns])
print('\nFeature checkpoint:', FEATURE_PATH)
print('Feature columns:', list(features.columns))
features.head(3)

Wrote c:\Users\52333\OneDrive - Bain\Desktop\CSE\T5HM\output\tables\model_features_whitespace_sc.csv | 2786 rows, 50 cols

Feature checkpoint: c:\Users\52333\OneDrive - Bain\Desktop\CSE\T5HM\output\tables\model_features_whitespace_sc.csv
Feature columns: ['anchor_id', 'anchor_source', 'anchor_name', 'address', 'lat', 'lng', 'catchment_side_m', 'is_inside_sc', 'candidate_sc_id', 'candidate_sc_sales_floor_m2', 'candidate_sc_sales_mn_jpy', 'prefecture_en', 'prefecture_jp', 'municipality_en', 'municipality_jp', 'density_tier', 'shinkansen_served_municipality', 'starbucks_in_500m_sq_cnt', 'tullys_in_500m_sq_cnt', 'tea_brands_in_500m_sq_cnt', 'mister_donut_in_500m_sq_cnt', 'competitors_in_500m_sq_cnt', 'gongcha_in_500m_sq_cnt', 'gongcha_inside_sc_in_500m_sq_cnt', 'sc_in_500m_sq_cnt', 'sc_sales_floor_m2_sum_in_500m_sq', 'sc_sales_mn_jpy_sum_in_500m_sq', 'stations_in_500m_sq_cnt', 'stations_passengers_2024_sum_in_500m_sq', 'shinkansen_in_500m_sq_cnt', 'schools_in_500m_sq_cnt', 'schools_univers

,anchor_id,anchor_source,anchor_name,address,lat,lng,catchment_side_m,is_inside_sc,candidate_sc_id,candidate_sc_sales_floor_m2,...,female_10_14_500m,female_15_19_500m,female_20_24_500m,beverages_per_household_500m,beverages_tea_per_household_500m,beverages_tea_drinks_per_household_500m,tea_share_of_beverages_500m,tea_drinks_share_of_beverages_500m,pop_density_per_km2_500m,female_share_of_pop_500m
0,wssc_1101012,whitespace_sc,サッポロファクトリー,北二条東4丁目1-2,43.064941,141.362678,500,1,1101012,44631.7,...,30,15,46,52058.257013,12118.384901,8404.606023,0.232785,0.161446,3636.0,0.594059
1,wssc_1101017,whitespace_sc,イオン札幌桑園ショッピングセンター,北8条西14-28,43.069413,141.333112,500,1,1101017,21581.0,...,39,53,32,59125.791261,13556.922958,9151.477536,0.229289,0.154780,5712.0,0.561625
2,wssc_1101021,whitespace_sc,東光ストア円山店,北一条西24-4-1,43.058708,141.320491,500,1,1101021,5237.0,...,65,81,73,52705.069613,12258.047445,8488.232630,0.232578,0.161052,10996.0,0.592579


In [33]:
# 8c. MLIT 1km mesh foot traffic — ingestion + caching.
# Heavy on first run (~5-10 min), parquet-cached afterwards.
# Delete `output/cache/foot_traffic_mesh.parquet` to force a fresh ingest.
#
# Compute optimizations (in order):
#   1. Stream-read 47 prefecture zips; only keep year + dayflag + timezone slices we need
#   2. Spatial pre-filter: drop any mesh outside the bounding box of all anchors (+ buffer).
#      Gongcha is concentrated in metros so this cuts ~90% of rows post-aggregation.
#   3. Aggregate 12 monthly readings -> annual mean per (mesh, year, dayflag, timezone)
#   4. Decode JIS 1km mesh code -> lat/lng (avoids attribute.zip download)
#   5. Pivot long -> wide so each mesh is one row with all time-slice columns
#   6. Cache the small final table (~5 MB) as parquet

import zipfile, io

OUT_CACHE = ROOT / 'output' / 'cache'
OUT_CACHE.mkdir(parents=True, exist_ok=True)
# Pickle (not parquet) to dodge a pandas-2.x + pyarrow-23 hotfix incompatibility.
FT_CACHE = OUT_CACHE / 'foot_traffic_mesh.pkl'
FT_DIR = DATA_DIR / 'foot traffic'

# Years + (dayflag, timezone) slices to keep.
#   dayflag : 0=全日 / 1=平日 / 2=休日
#   timezone: 0=終日 / 1=昼 / 2=夜
FT_YEARS = (2019, 2021)
FT_SLICES = {
    (0, 0): 'all_alltime',         # main foot-traffic metric
    (1, 1): 'weekday_daytime',     # commuter / business
    (2, 1): 'weekend_daytime',     # leisure / shopping
    (0, 2): 'all_nighttime',       # residential-population proxy
}

# Bounding-box buffer around anchors: keep meshes within ~7km (covers the 5km ring).
ANCHOR_BBOX_BUFFER_DEG = 0.07

def decode_mesh1km_to_latlng(mesh_ids):
    """Decode JIS X 0410 standard 1km mesh code (8 digits) to centroid (lat, lng).
    Cell pattern: 1st mesh 80x100km -> 2nd mesh /8 -> 3rd mesh /10."""
    m = pd.to_numeric(mesh_ids, errors='coerce').astype('Int64')
    A = (m // 10000000).astype(float)
    B = ((m // 1000000) % 10).astype(float)
    C = ((m // 100000) % 10).astype(float)
    D = ((m // 10000) % 10).astype(float)
    E = ((m // 1000) % 10).astype(float)
    F = ((m // 100) % 10).astype(float)
    G = ((m // 10) % 10).astype(float)
    H = (m % 10).astype(float)
    # 1st mesh: lat = AB/1.5, lng = CD+100
    # 2nd mesh (1/8 of 1st): /12 for lat (5min), /8 for lng (7.5min)
    # 3rd mesh (1/10 of 2nd): /120 for lat (30sec), /80 for lng (45sec)
    lat_sw = (A * 10 + B) / 1.5 + E / 12 + G / 120
    lng_sw = (C * 10 + D) + 100 + F / 8 + H / 80
    return lat_sw + 0.5 / 120, lng_sw + 0.5 / 80  # add half-cell to get center

def ingest_mesh_foot_traffic():
    """Load + filter + aggregate. Returns wide DataFrame keyed by mesh."""
    from tqdm.auto import tqdm

    lat_min = anchors['lat'].min() - ANCHOR_BBOX_BUFFER_DEG
    lat_max = anchors['lat'].max() + ANCHOR_BBOX_BUFFER_DEG
    lng_min = anchors['lng'].min() - ANCHOR_BBOX_BUFFER_DEG
    lng_max = anchors['lng'].max() + ANCHOR_BBOX_BUFFER_DEG
    print(f'Anchor bbox  : lat [{lat_min:.2f}, {lat_max:.2f}], lng [{lng_min:.2f}, {lng_max:.2f}]')

    pref_zips = sorted(FT_DIR.glob('monthly_mdp_mesh1km_*.zip'))
    print(f'Prefecture zips: {len(pref_zips)}')

    chunks = []
    for zpath in tqdm(pref_zips, desc='prefectures'):
        with zipfile.ZipFile(zpath) as outer:
            inner_zips = [n for n in outer.namelist() if n.endswith('.csv.zip')]
            for n in inner_zips:
                # path like '01/2019/03/monthly_mdp_mesh1km.csv.zip'
                year = int(n.split('/')[1])
                if year not in FT_YEARS:
                    continue
                with outer.open(n) as f:
                    data = f.read()
                with zipfile.ZipFile(io.BytesIO(data)) as inner:
                    with inner.open(inner.namelist()[0]) as cf:
                        df = pd.read_csv(cf, encoding='cp932',
                                         usecols=['mesh1kmid', 'year', 'month',
                                                  'dayflag', 'timezone', 'population'])
                # Keep only the slices we care about (vectorized via dayflag*10+timezone)
                wanted = {k[0] * 10 + k[1] for k in FT_SLICES}
                df = df[(df['dayflag'] * 10 + df['timezone']).isin(wanted)]
                if len(df):
                    chunks.append(df)

    raw = pd.concat(chunks, ignore_index=True)
    print(f'\nMonthly rows after slice filter: {len(raw):,}')

    # 12-month -> annual mean per (mesh, year, dayflag, timezone)
    annual = (raw.groupby(['mesh1kmid', 'year', 'dayflag', 'timezone'])['population']
                 .mean().reset_index()
                 .rename(columns={'population': 'pop_avg'}))
    print(f'Annual aggregated rows         : {len(annual):,}')

    # Decode mesh code -> coords, then drop meshes outside anchor bbox
    annual['lat'], annual['lng'] = decode_mesh1km_to_latlng(annual['mesh1kmid'])
    annual = annual[(annual['lat'] >= lat_min) & (annual['lat'] <= lat_max)
                    & (annual['lng'] >= lng_min) & (annual['lng'] <= lng_max)].copy()
    print(f'After anchor-bbox filter       : {len(annual):,}')

    # Long -> wide (vectorized slice-name lookup)
    slice_lookup = {k[0] * 10 + k[1]: v for k, v in FT_SLICES.items()}
    annual['slice'] = (annual['dayflag'] * 10 + annual['timezone']).map(slice_lookup)
    annual['col'] = 'pop_' + annual['year'].astype(str) + '_' + annual['slice']
    wide = (annual.pivot_table(index=['mesh1kmid', 'lat', 'lng'],
                               columns='col', values='pop_avg')
                  .reset_index())
    wide.columns.name = None
    print(f'Wide mesh table                : {wide.shape[0]:,} meshes × {wide.shape[1] - 3} pop cols')
    return wide

RUN_FOOT_TRAFFIC = True  # flip False to skip foot traffic entirely
if not RUN_FOOT_TRAFFIC:
    foot_traffic_mesh = pd.DataFrame(columns=['mesh1kmid', 'lat', 'lng'])
    print('Skipping foot traffic (RUN_FOOT_TRAFFIC=False)')
elif FT_CACHE.exists():
    foot_traffic_mesh = pd.read_pickle(FT_CACHE)
    print(f'Loaded cached: {len(foot_traffic_mesh):,} meshes × {foot_traffic_mesh.shape[1]} cols ({FT_CACHE.name})')
else:
    foot_traffic_mesh = ingest_mesh_foot_traffic()
    foot_traffic_mesh.to_pickle(FT_CACHE)
    print(f'\nCached -> {FT_CACHE}')


Loaded cached: 196,470 meshes × 11 cols (foot_traffic_mesh.pkl)


In [34]:
# 8d. Foot traffic feature block + merge into model_features_full.csv.
# The mesh table is treated like any other POI table; build_block runs sum-in-square
# and sum-in-ring on each population time-slice column.
#
# Whitespace grid scaling: if ANCHOR_MODE='whitespace_grid' produces >5000 anchors,
# CHUNK_SIZE_FT chunks the haversine matrix per anchor batch to bound memory.
CHUNK_SIZE_FT = 5000

if len(foot_traffic_mesh):
    ft_pois = foot_traffic_mesh.rename(columns={'mesh1kmid': 'poi_id'}).copy()
    ft_pois['poi_id'] = 'mesh_' + ft_pois['poi_id'].astype(str)
    pop_cols = [c for c in ft_pois.columns if c.startswith('pop_')]
    print(f'Foot traffic POI: {len(ft_pois):,} meshes, {len(pop_cols)} value cols')

    name = 'foot_traffic'
    if (MODEL_FEATURES_ONLY_FOR_WHITESPACE and REQUIRED_DEPLOYMENT_FEATURES is not None):
        pop_cols_needed = [c for c in pop_cols if f'{name}_{c}_sum_in_{side_tag}' in REQUIRED_DEPLOYMENT_FEATURES]
        print(f'Foot traffic model-only mode: {len(pop_cols_needed)} square-sum cols needed')
    else:
        pop_cols_needed = pop_cols

    if len(anchors) > CHUNK_SIZE_FT:
        from tqdm.auto import tqdm
        sub_blocks = []
        for s in tqdm(range(0, len(anchors), CHUNK_SIZE_FT), desc='ft anchor chunks'):
            sub = anchors.iloc[s:s + CHUNK_SIZE_FT]
            if MODEL_FEATURES_ONLY_FOR_WHITESPACE and REQUIRED_DEPLOYMENT_FEATURES is not None:
                sq_sum = sum_pois_in_square(sub, ft_pois, pop_cols_needed).rename(
                    columns={f'{vc}_sum': f'{name}_{vc}_sum_in_{side_tag}' for vc in pop_cols_needed})
                sub_blocks.append(sq_sum)
            else:
                sq = count_pois_in_square(sub, ft_pois).rename(columns={'cnt': f'{name}_in_{side_tag}_cnt'})
                rings = count_pois_in_rings(sub, ft_pois, RINGS_KM, name)
                sq_sum = sum_pois_in_square(sub, ft_pois, pop_cols).rename(
                    columns={f'{vc}_sum': f'{name}_{vc}_sum_in_{side_tag}' for vc in pop_cols})
                ring_sum = sum_pois_in_rings(sub, ft_pois, pop_cols, RINGS_KM, name)
                sub_blocks.append(sq.merge(rings, on='anchor_id', how='outer')
                                    .merge(sq_sum, on='anchor_id', how='outer')
                                    .merge(ring_sum, on='anchor_id', how='outer'))
        ft_block = pd.concat(sub_blocks, ignore_index=True)
    else:
        ft_block = build_block(name, ft_pois, value_cols=pop_cols_needed)

    feature_blocks['foot_traffic'] = ft_block
    print(f'Foot traffic block: {ft_block.shape[0]} anchors × {ft_block.shape[1] - 1} feature cols')
else:
    print('No foot traffic data; skipping block')

# Re-assemble FULL features (everything in 8a + foot traffic) and save second checkpoint
features_full = anchors.copy()
for src, block in feature_blocks.items():
    features_full = features_full.merge(block, on='anchor_id', how='left')
cnt_cols = [c for c in features_full.columns if '_cnt_' in c]
for c in cnt_cols:
    features_full[c] = features_full[c].fillna(0).astype(int)

csv_write(features_full, FEATURE_FULL_PATH,
          sanitize_cols=[c for c in ['anchor_name','address'] if c in features_full.columns])
print(f'\nSaved {FEATURE_FULL_PATH.name}: {features_full.shape[0]} anchors × {features_full.shape[1]} features')
ft_cols_added = sum(1 for c in features_full.columns if 'foot_traffic' in c)
print(f'  foot traffic cols added: {ft_cols_added}')


Foot traffic POI: 196,470 meshes, 8 value cols
Foot traffic model-only mode: 8 square-sum cols needed
  foot_traffic        : 2786 anchors x  8 feature cols
Foot traffic block: 2786 anchors × 8 feature cols
Wrote c:\Users\52333\OneDrive - Bain\Desktop\CSE\T5HM\output\tables\model_features_whitespace_sc_full.csv | 2786 rows, 58 cols

Saved model_features_whitespace_sc_full.csv: 2786 anchors × 58 features
  foot traffic cols added: 8


## 9. Sanity check — `model_features_full.csv`

Three lenses on the final feature table:

- **9a** — shape, column-group breakdown (which families of features made it in)
- **9b** — per-column missing rate, grouped by family (catches silent join misses)
- **9c** — summary statistics on representative cols (catches unit-mismatch / scale anomalies)
- **9d** — attribute-checklist coverage vs `T5HM_AIS_data_prep_list 1.xlsx`

In [35]:
# 9a. Load + column-group overview.
import re

mf = pd.read_csv(FEATURE_FULL_PATH, encoding='utf-8-sig')
print(f'Loaded {FEATURE_FULL_PATH.name} : {mf.shape[0]} rows × {mf.shape[1]} cols')

# Bucket each column into a feature family for readability
def bucket_col(c):
    if c in {'anchor_id', 'anchor_source', 'anchor_name', 'address', 'lat', 'lng',
            'store_type', 'open_year', 'open_years', 'catchment_side_m',
            'store_area_sqft', 'store_area_m2', 'seats'}:
        return '0. anchor identity / store attrs'
    if c in {'prefecture_en', 'prefecture_jp', 'municipality_en', 'municipality_jp',
            'density_tier', 'shinkansen_served_municipality'}:
        return '1. region / admin tags'
    if c.startswith(('pop_total_', 'pop_female_', 'households_', 'household_income_',
                    'beverages_', 'pop_age_', 'female_', 'women_')):
        return '2. ESRI demographics (catchment)'
    if c.startswith('foot_traffic_'):
        return '8. MLIT mesh foot traffic'
    if c.startswith(('starbucks_', 'tullys_', 'tea_brands_', 'competitors_')):
        return '3. competitors (per-brand + combined)'
    if c.startswith('gongcha_') or c in {'nearest_gongcha_id', 'nearest_gongcha_dist_km'}:
        return '4. Gongcha self (cannibalization)'
    if c.startswith('sc_'):
        return '5. shopping centers'
    if c.startswith(('stations_', 'nearest_stations_')):
        return '6. railway stations + passengers'
    if c.startswith(('shinkansen_', 'nearest_shinkansen_')):
        return '7. Shinkansen'
    if c.startswith('schools_'):
        return '9. schools (overall + slices)'
    return 'Z. unclassified'

buckets = pd.Series({c: bucket_col(c) for c in mf.columns})
group_counts = buckets.value_counts().sort_index()
print('\nFeature columns by family:')
for grp, n in group_counts.items():
    print(f'  {grp:50s} : {n:>3} cols')
print(f'  {"TOTAL":50s} : {len(mf.columns):>3} cols')

Loaded model_features_whitespace_sc_full.csv : 2786 rows × 58 cols

Feature columns by family:
  0. anchor identity / store attrs                   :   7 cols
  1. region / admin tags                             :   6 cols
  2. ESRI demographics (catchment)                   :  14 cols
  3. competitors (per-brand + combined)              :   4 cols
  4. Gongcha self (cannibalization)                  :   2 cols
  5. shopping centers                                :   3 cols
  6. railway stations + passengers                   :   2 cols
  7. Shinkansen                                      :   1 cols
  8. MLIT mesh foot traffic                          :   8 cols
  9. schools (overall + slices)                      :   3 cols
  Z. unclassified                                    :   8 cols
  TOTAL                                              :  58 cols


In [36]:
# 9b. Missing-rate per column, grouped by family.
# A clean run should have:
#   * 0% missing on count/sum cols (we fill _cnt_ with 0)
#   * a handful of NaN on nearest_*_dist_km only when the POI table is empty
#   * 1-30% missing on store_area_sqft / store_area_m2 / seats (client data partial)
#   * any other column with >5% missing is a red flag

miss = mf.isna().mean().rename('miss_rate')
miss_df = pd.DataFrame({'family': buckets, 'miss_rate': miss}).reset_index().rename(columns={'index': 'col'})

print('=== Per-family missing-rate summary ===')
fam_stat = (miss_df.groupby('family')['miss_rate']
                  .agg(['count', 'mean', 'max']).round(3)
                  .sort_index())
print(fam_stat.to_string())

print('\n=== Columns with > 5% missing (sorted by miss_rate) ===')
problem = miss_df[miss_df['miss_rate'] > 0.05].sort_values('miss_rate', ascending=False)
if len(problem):
    print(problem.to_string(index=False))
else:
    print('  (none — all cols have <=5% missing)')

print('\n=== Columns that are entirely NaN (full failure) ===')
zeros = miss_df[miss_df['miss_rate'] == 1.0]
if len(zeros):
    print(zeros.to_string(index=False))
else:
    print('  (none)')

=== Per-family missing-rate summary ===
                                       count   mean    max
family                                                    
0. anchor identity / store attrs           7  0.000  0.000
1. region / admin tags                     6  0.000  0.001
2. ESRI demographics (catchment)          14  0.012  0.043
3. competitors (per-brand + combined)      4  0.000  0.000
4. Gongcha self (cannibalization)          2  0.000  0.000
5. shopping centers                        3  0.000  0.000
6. railway stations + passengers           2  0.000  0.000
7. Shinkansen                              1  0.000  0.000
8. MLIT mesh foot traffic                  8  0.000  0.000
9. schools (overall + slices)              3  0.000  0.000
Z. unclassified                            8  0.127  0.931

=== Columns with > 5% missing (sorted by miss_rate) ===
                      col          family  miss_rate
candidate_sc_sales_mn_jpy Z. unclassified   0.931443

=== Columns that are entirely

In [37]:
# 9c. Summary stats for representative columns in each family.
# Expected ballpark sanity (Japan urban anchors, 500m sq + 1/3/5km rings):
#   pop_total_500m         : O(1K-30K)   -- ESRI residential
#   household_income_avg   : ~5,000-9,000 (000 JPY)
#   beverages_total_500m   : O(100K-3000K) JPY (HH spend × #HH × 1 year)
#   *_cnt_5km              : urban anchors easily 10-100 competitors
#   stations_passengers_2024_sum_5km : O(100K-30M) daily passengers
#   foot_traffic_pop_2021_all_alltime_sum_1km : O(1K-50K) avg presence
#   sc_sales_mn_jpy_sum_5km : O(10K-200K) mn JPY (only a few SCs per ring)

probe_cols = [
    'store_area_m2', 'seats',
    'pop_total_500m', 'pop_female_total_500m', 'women_10_29_500m',
    'household_income_avg_500m', 'beverages_total_500m', 'beverages_tea_500m',
    'starbucks_in_500m_sq_cnt', 'starbucks_cnt_5km',
    'competitors_cnt_1km', 'competitors_cnt_5km',
    'gongcha_cnt_3km', 'nearest_gongcha_dist_km',
    'sc_in_500m_sq_cnt', 'sc_sales_mn_jpy_sum_5km',
    'stations_cnt_1km', 'stations_passengers_2024_sum_1km',
    'shinkansen_cnt_5km', 'nearest_shinkansen_dist_km', 'shinkansen_served_municipality',
    'schools_university_cnt_3km', 'schools_high_school_cnt_3km',
]
ft_cols_alltime = [c for c in mf.columns if 'foot_traffic_pop_2021_all_alltime' in c]
probe_cols += ft_cols_alltime

present = [c for c in probe_cols if c in mf.columns]
absent  = [c for c in probe_cols if c not in mf.columns]

if absent:
    print(f'(probe cols not in df: {absent})\n')

stats = mf[present].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).T
stats['n_zero'] = (mf[present] == 0).sum().values
stats['n_null'] = mf[present].isna().sum().values
stats = stats[['count', 'n_null', 'n_zero', 'min', '25%', '50%', '75%', '95%', 'max', 'mean']]
print('=== Probe statistics (Japan, n=186 anchors) ===')
print(stats.round(2).to_string())

(probe cols not in df: ['store_area_m2', 'seats', 'women_10_29_500m', 'starbucks_cnt_5km', 'competitors_cnt_1km', 'competitors_cnt_5km', 'gongcha_cnt_3km', 'nearest_gongcha_dist_km', 'sc_sales_mn_jpy_sum_5km', 'stations_cnt_1km', 'stations_passengers_2024_sum_1km', 'shinkansen_cnt_5km', 'nearest_shinkansen_dist_km', 'schools_university_cnt_3km', 'schools_high_school_cnt_3km'])

=== Probe statistics (Japan, n=186 anchors) ===
                                                   count  n_null  n_zero  min         25%         50%          75%          95%           max         mean
pop_total_500m                                    2786.0       0     118  0.0      202.50       536.0      1339.25      3172.25  9.613000e+03       923.95
pop_female_total_500m                             2786.0       0     126  0.0      103.00       273.0       700.75      1694.00  5.000000e+03       485.57
household_income_avg_500m                         2786.0       0     120  0.0  4360774.25   4837108.5   53

In [38]:
# 9d. Attribute coverage check against T5HM_AIS_data_prep_list 1.xlsx (Japan column).
# Mapping is hardcoded so this works offline and stays as project documentation.
# Status:
#   OK      = all requested cuts available in model_features_full.csv
#   PARTIAL = requirement partly met (e.g. brand-level OK, SC inside/outside split missing)
#   GAP     = no column in model_features_full.csv covering this

CHECKLIST = [
    # category, requirement (from xlsx Japan column), status, covered_by, note
    ('Store attributes', 'Region (Tokyo/Osaka/focus cities/other)',          'OK',
        'prefecture_en, prefecture_jp, density_tier',
        ''),
    ('Store attributes', 'Store area / size (VDR Sq. Ft.)',                  'OK',
        'store_area_sqft, store_area_m2',
        '99% coverage on Japan-Open subset'),
    ('Store attributes', '# of seats / seat capacity (VDR Seats)',           'OK',
        'seats',
        '68% coverage (148/217 Japan-Open stores)'),
        ('Store attributes', 'Store format (inside-SC flag, Biz Zone, Location Type)', 'OK',
        'is_inside_sc, biz_zone, location_type',
        '100% coverage; IS/RS split = 206 inside SC vs 11 roadside (Japan-Open)'),

    ('Demographics (catchment)', 'Populations all ages/genders',             'OK',
        'pop_total_500m, pop_female_total_500m',
        ''),
    ('Demographics (catchment)', 'Women teens & 20s (separate cuts)',        'OK',
        'female_15_19, female_20_24, female_25_29, women_10_29_500m',
        ''),
    ('Demographics (catchment)', 'Foot traffic (1km grid)',                  'OK',
        'foot_traffic_pop_2019/2021_* (all_alltime, weekday_daytime, weekend_daytime, all_nighttime) x (500m sq + 1/3/5km)',
        'MLIT 1km mesh; 2019 baseline + 2021 latest'),
    ('Demographics (catchment)', 'Station passengers (S12 乗降客数)',          'OK',
        'stations_passengers_2019/2024_sum_*',
        'sum across 500m sq + 1/3/5km rings'),
    ('Demographics (catchment)', 'Household income (ESRI)',                  'OK',
        'household_income_avg_500m',
        ''),
    ('Demographics (catchment)', 'Spending for beverages / tea',             'OK',
        'beverages_total_500m, beverages_tea_500m, beverages_tea_drinks_500m',
        ''),

    ('Competitors (catchment)', '# of Genuine (Gongcha) — total',            'OK',
        'gongcha_in_500m_sq_cnt, gongcha_cnt_1/3/5km, nearest_gongcha_dist_km',
        'exclude_self=True (no self-counting)'),
        ('Competitors (catchment)', '# of Genuine inside SC vs outside SC',      'OK',
        'gongcha_inside_sc_cnt_1/3/5km, gongcha_outside_sc_cnt_1/3/5km, gongcha_cnt_* (total)',
        'Split uses VDR IS/RS flag (ground truth, 100% coverage). Pivots per attribute-list spec.'),
    ('Competitors (catchment)', '# of Starbucks',                            'OK',
        'starbucks_in_500m_sq_cnt, starbucks_cnt_1/3/5km',
        ''),
    ('Competitors (catchment)', 'CHA BAR / Pearl Lady / Bull Pulu / 春水堂 / The Alley', 'OK',
        'tea_brands_* (combined across these 5 brands)',
        '99/99 geocoded with score>=80'),
    ('Competitors (catchment)', "Tully's",                                   'OK',
        'tullys_*',
        ''),
        ('Competitors (catchment)', 'Mister Donut',                              'OK',
        'mister_donut_*',
        '1,073 stores from 260520_T5HM_misterDonut_JP.xlsx; 100%% lat/lng coverage (no geocoding needed)'),
        ('Competitors (catchment)', 'Per-competitor inside-SC vs outside-SC',    'OUT OF SCOPE',
        '-',
        'Per discussion: only Gongcha (Genuine) needs the inside/outside split; competitors keep brand-total only'),

    ('Others (catchment)', '# of shopping malls + SC sales',                 'OK',
        'sc_in_500m_sq_cnt, sc_cnt_1/3/5km, sc_sales_mn_jpy_sum_*, sc_sales_floor_m2_sum_*',
        ''),
    ('Others (catchment)', '# of schools (overall + HS/univ/vocational)',    'OK',
        'schools_*, schools_high_school_*, schools_university_*, schools_vocational_*',
        ''),
    ('Others (catchment)', 'Shinkansen city flag + catchment Shinkansen station', 'OK',
        'shinkansen_served_municipality, shinkansen_in_500m_sq_cnt, shinkansen_cnt_1/3/5km, nearest_shinkansen_dist_km',
        ''),
    ('Others (catchment)', 'Private Consumption',                            'PARTIAL',
        'beverages_* (food/drink subset only)',
        'xlsx marks "/" for Japan -- treating as N/A unless client clarifies'),
    ('Others (catchment)', 'Transportation',                                 'PARTIAL',
        'stations_passengers_*, shinkansen_*',
        'xlsx marks "/" for Japan -- station passengers proxies this'),

    ('Y (performance)', 'Sales / EBITDA / per-sqm',                          'GAP',
        '-',
        'Client to provide; out of scope for X-side data prep'),
]

cl = pd.DataFrame(CHECKLIST, columns=['category', 'requirement', 'status', 'covered_by', 'note'])
print(f'=== Attribute checklist coverage ({len(cl)} requirements) ===\n')
summary = cl['status'].value_counts()
for st in ['OK', 'PARTIAL', 'GAP']:
    print(f'  {st:8s}: {summary.get(st, 0)}')
print()
with pd.option_context('display.max_colwidth', 100, 'display.width', 200):
    print(cl.to_string(index=False))

=== Attribute checklist coverage (23 requirements) ===

  OK      : 19
  PARTIAL : 2
  GAP     : 1

                category                                            requirement       status                                                                                                        covered_by                                                                                                     note
        Store attributes                Region (Tokyo/Osaka/focus cities/other)           OK                                                                        prefecture_en, prefecture_jp, density_tier                                                                                                         
        Store attributes                        Store area / size (VDR Sq. Ft.)           OK                                                                                    store_area_sqft, store_area_m2                                                                      

## 10. Build Y targets (sales + EBITDA)

Targets are written to `output/tables/y_targets.csv` and consumed by `04_modeling.ipynb`.

### A) Sales-based targets (from monthly POS)
- Source: `Sheet2_Database` of `260513_global_store_address_vshare_..._vup.xlsx` (84 monthly POS columns, 2019-01 → 2025-12).
- Filter `Region == Japan`, drop months where sales ≤ 0 / null.
- Drop first 6 valid months per store (ramp-up exclusion).

| target | definition |
|---|---|
| `monthly_sales_avg_alltime_jpy` | mean of all post-ramp valid months |
| `monthly_sales_avg_l6m_jpy` | mean of each store's own latest 6 post-ramp valid months |

### B) EBITDA target (from AUV workbook)
- Source: `Database - Local Currency` in `260520_AUV分析_v12.xlsb`.
- Use latest full-year EBITDA and sales summary columns, and map store to anchor by `anchor_id = gc_{No.}`.
- Keep only `Region == Japan`; deduplicate duplicate `No.` rows by taking the row with non-null EBITDA first.

| target | definition |
|---|---|
| `ebitda_2025_local` | latest-year EBITDA (local currency) |
| `ebitda_margin_2025` | `ebitda_2025_local / sales_2025_local` |

Notes:
- EBITDA coverage is lower than POS monthly sales; this is expected due to missing financial records in AUV.
- The anchor filter still controls the modeling universe (186 anchors).

In [112]:
# 10a. Loader: parse Japan monthly POS sales out of Sheet2_Database.
import openpyxl

GONGCHA_MASTER_XLSX = DATA_DIR / 'internal' / (
    '260513_global_store_address_vshare_ジオコーディング修正_手作業追加_vup.xlsx')

# Excel layout (python 0-indexed within each iter_rows tuple — col A is blank):
#   r[1]  = No.            r[2]  = Store name      r[4]  = Region        r[8]  = Currency
#   r[18] = Open/Close     r[19] = Open date       r[20] = Open year
#   r[48]..r[131]          = 84 monthly POS values, 2019-01 .. 2025-12  (column-row 10 has POS / Month # labels)
SALES_SHEET = 'Sheet2_Database'
SALES_DATA_START_ROW = 11
SALES_MONTH_FIRST_IDX = 48
SALES_MONTH_LAST_IDX  = 131
SALES_MONTHS = pd.date_range('2019-01-01',
                             periods=SALES_MONTH_LAST_IDX - SALES_MONTH_FIRST_IDX + 1,
                             freq='MS')


def load_gongcha_monthly_sales(xlsx_path=GONGCHA_MASTER_XLSX,
                               sheet=SALES_SHEET,
                               region_filter='Japan'):
    """Return long-format DataFrame: anchor_id, gongcha_no, year_month, sales_jpy.

    - Filters to a single region (default Japan).
    - Skips months where sales are null, blank, or <= 0 (treated as 'not operating').
    """
    wb = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb[sheet]
    records = []
    for row in ws.iter_rows(min_row=SALES_DATA_START_ROW, values_only=True):
        no, region = row[1], row[4]
        if region != region_filter or no is None:
            continue
        monthly = row[SALES_MONTH_FIRST_IDX: SALES_MONTH_LAST_IDX + 1]
        for m_idx, v in enumerate(monthly):
            if v is None or (isinstance(v, str) and v.strip() == ''):
                continue
            try:
                sales = float(v)
            except (TypeError, ValueError):
                continue
            if sales <= 0:
                continue
            records.append({
                'anchor_id': f'gc_{int(no)}',
                'gongcha_no': int(no),
                'year_month': SALES_MONTHS[m_idx],
                'sales_jpy': sales,
            })
    return pd.DataFrame(records)


sales_long = load_gongcha_monthly_sales()
print(f'Monthly sales rows : {len(sales_long):,}')
print(f'Unique stores      : {sales_long["anchor_id"].nunique()}')
print(f'Date range         : {sales_long["year_month"].min():%Y-%m} .. '
      f'{sales_long["year_month"].max():%Y-%m}')
print(f'Median month sale  : JPY {sales_long["sales_jpy"].median():,.0f}')

Monthly sales rows : 10,120
Unique stores      : 262
Date range         : 2019-01 .. 2025-12
Median month sale  : JPY 7,618,120


In [113]:
# 10b. Build Y targets: sales (monthly POS) + EBITDA (AUV xlsb)
RAMP_UP_MONTHS = 6
ALLTIME_MIN_MONTHS = 2
L6M_MIN_MONTHS = 2

# ---- A) Sales targets from monthly POS ----
sl = sales_long.sort_values(['anchor_id', 'year_month']).copy()
sl['rank_from_open'] = sl.groupby('anchor_id').cumcount() + 1
sl_post = sl[sl['rank_from_open'] > RAMP_UP_MONTHS].copy()

sl_post = sl_post.sort_values(['anchor_id', 'year_month'], ascending=[True, False])
sl_post['rank_from_latest'] = sl_post.groupby('anchor_id').cumcount() + 1

y_all = (sl_post.groupby('anchor_id')['sales_jpy']
         .agg(monthly_sales_avg_alltime_jpy='mean', n_months_alltime='count')
         .reset_index())

l6m_slice = sl_post[sl_post['rank_from_latest'] <= 6]
y_l6 = (l6m_slice.groupby('anchor_id').agg(
            monthly_sales_avg_l6m_jpy=('sales_jpy', 'mean'),
            n_months_l6m=('sales_jpy', 'count'),
            l6m_first_month=('year_month', 'min'),
            l6m_last_month=('year_month', 'max'))
        .reset_index())

y_targets = y_all.merge(y_l6, on='anchor_id', how='outer')
y_targets['n_months_alltime'] = y_targets['n_months_alltime'].fillna(0).astype(int)
y_targets['n_months_l6m']     = y_targets['n_months_l6m'].fillna(0).astype(int)
y_targets.loc[y_targets['n_months_alltime'] < ALLTIME_MIN_MONTHS,
              'monthly_sales_avg_alltime_jpy'] = np.nan
y_targets.loc[y_targets['n_months_l6m'] < L6M_MIN_MONTHS,
              'monthly_sales_avg_l6m_jpy'] = np.nan

# ---- B) EBITDA targets from AUV workbook (.xlsb) ----
# Layout source (header row index=8 in Excel):
#   col 3   = No.
#   col 6   = Region
#   col 20  = Open/Close
#   col 230 = Sales Total (latest-year summary)
#   col 239 = EBITDA (latest-year summary)
#   col 501..504 = address / lat / lon / prefecture (for QA only)
AUV_XLSB = DATA_DIR / '260520_AUV分析_v12.xlsb'
AUV_SHEET = 'Database - Local Currency'

def load_japan_auv_financials(xlsb_path=AUV_XLSB, sheet=AUV_SHEET):
    raw = pd.read_excel(xlsb_path, sheet_name=sheet, engine='pyxlsb', header=None)
    d = raw.iloc[9:].copy()   # data rows start after the header block

    sel = {
        3: 'store_no',
        4: 'store_name_auv',
        6: 'region',
        20: 'open_close',
        230: 'sales_2025_local',
        239: 'ebitda_2025_local',
        501: 'full_address_auv',
        502: 'latitude_auv',
        503: 'longitude_auv',
        504: 'prefecture_auv',
    }
    out = pd.DataFrame({name: d.iloc[:, idx] for idx, name in sel.items() if idx < d.shape[1]})

    # Japan only + stable anchor mapping
    out = out[out['region'].astype(str).str.strip().eq('Japan')].copy()
    out['store_no'] = pd.to_numeric(out['store_no'], errors='coerce').astype('Int64')
    out = out[out['store_no'].notna()].copy()
    out['anchor_id'] = 'gc_' + out['store_no'].astype(str)

    # numeric coercion
    for c in ['sales_2025_local', 'ebitda_2025_local', 'latitude_auv', 'longitude_auv']:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')

    # one row per anchor_id: prioritize non-null EBITDA, then non-null sales
    out['_score'] = out['ebitda_2025_local'].notna().astype(int) * 2 + out['sales_2025_local'].notna().astype(int)
    out = out.sort_values(['anchor_id', '_score'], ascending=[True, False]).drop_duplicates('anchor_id', keep='first')
    out = out.drop(columns=['_score'])

    out['ebitda_margin_2025'] = out['ebitda_2025_local'] / out['sales_2025_local']
    out.loc[(out['sales_2025_local'].isna()) | (out['sales_2025_local'] <= 0), 'ebitda_margin_2025'] = np.nan
    return out

au_fin = load_japan_auv_financials()

# Merge EBITDA fields into y_targets by anchor_id
y_targets = y_targets.merge(
    au_fin[['anchor_id', 'sales_2025_local', 'ebitda_2025_local', 'ebitda_margin_2025']],
    on='anchor_id', how='left'
)

Y_PATH = OUT_TABLES / 'y_targets.csv'
y_targets.to_csv(Y_PATH, index=False, encoding='utf-8-sig')

print(f'Wrote {Y_PATH} | {len(y_targets)} stores (across all Japan)')
print(f'Ramp-up months excluded     : first {RAMP_UP_MONTHS} valid months of each store')
print(f'Minimum post-ramp months    : {ALLTIME_MIN_MONTHS} for all-time, '
      f'{L6M_MIN_MONTHS} for L6M (out of personal last 6)')
print('\nL6M window per-store (range of l6m_last_month):')
print(f'  newest        : {y_targets["l6m_last_month"].max()}')
print(f'  oldest        : {y_targets["l6m_last_month"].min()}')

print('\nCoverage (before anchor filtering in modeling notebook):')
print(f'  sales all-time  : {y_targets["monthly_sales_avg_alltime_jpy"].notna().sum()} / {len(y_targets)}')
print(f'  sales l6m       : {y_targets["monthly_sales_avg_l6m_jpy"].notna().sum()} / {len(y_targets)}')
print(f'  EBITDA 2025     : {y_targets["ebitda_2025_local"].notna().sum()} / {len(y_targets)}')
print(f'  EBITDA margin   : {y_targets["ebitda_margin_2025"].notna().sum()} / {len(y_targets)}')

print('\nTarget distribution (Japan):')
print(y_targets[['monthly_sales_avg_alltime_jpy', 'monthly_sales_avg_l6m_jpy',
                'ebitda_2025_local', 'ebitda_margin_2025',
                'n_months_alltime', 'n_months_l6m']]
      .describe(percentiles=[.1, .5, .9]).round(3).to_string())

Wrote c:\Users\52333\OneDrive - Bain\Desktop\CSE\T5HM\output\tables\y_targets.csv | 242 stores (across all Japan)
Ramp-up months excluded     : first 6 valid months of each store
Minimum post-ramp months    : 2 for all-time, 2 for L6M (out of personal last 6)

L6M window per-store (range of l6m_last_month):
  newest        : 2025-12-01 00:00:00
  oldest        : 2020-04-01 00:00:00
  most-common 5 :
l6m_last_month
2025-12-01    190
2024-03-01      7
2023-01-01      4
2020-04-01      3
2025-08-01      3

Target distribution (Japan, before anchor filter):
       monthly_sales_avg_alltime_jpy  monthly_sales_avg_l6m_jpy  n_months_alltime  n_months_l6m
count                          236.0                      236.0             242.0         242.0
mean                       8097978.0                  9697388.0              36.0           6.0
std                        2917304.0                  3962665.0              24.0           1.0
min                         366046.0                   2

In [114]:
# 10c. Coverage + sanity vs the 186 modeling anchors.
anchors_check = pd.read_csv(OUT_TABLES / 'model_features_full.csv',
                            encoding='utf-8-sig',
                            usecols=['anchor_id', 'anchor_name', 'open_year', 'open_years'])

cov = anchors_check.merge(y_targets, on='anchor_id', how='left')
print(f'Anchors                                                            : {len(cov)}')
print(f'  with all-time Y (post-ramp, >= {ALLTIME_MIN_MONTHS} months)                          : '
      f'{cov["monthly_sales_avg_alltime_jpy"].notna().sum()}')
print(f'  with L6M Y (personal last 6 post-ramp, >= {L6M_MIN_MONTHS} months)              : '
      f'{cov["monthly_sales_avg_l6m_jpy"].notna().sum()}')
print(f'  with EBITDA Y (AUV latest-year)                                                  : '
      f'{cov["ebitda_2025_local"].notna().sum()}')
print(f'  with EBITDA margin                                                               : '
      f'{cov["ebitda_margin_2025"].notna().sum()}')

# Flag stores using a smaller-than-typical sample so the modeler can downweight if desired
thin_sample = cov[(cov['n_months_alltime'] < 6) & (cov['n_months_alltime'] > 0)]
if len(thin_sample):
    print(f'\nAnchors with thin sample (1-5 post-ramp months — newly opened, treat with caution):')
    print(thin_sample[['anchor_id', 'anchor_name', 'open_year', 'n_months_alltime',
                       'l6m_first_month', 'l6m_last_month']].to_string(index=False))

missing_all = cov[cov['monthly_sales_avg_alltime_jpy'].isna()]
if len(missing_all):
    print(f'\nAnchors missing all-time Y ({len(missing_all)}):')
    print(missing_all[['anchor_id', 'anchor_name', 'n_months_alltime']]
          .head(10).to_string(index=False))

missing_l6m = cov[cov['monthly_sales_avg_l6m_jpy'].isna()]
if len(missing_l6m):
    print(f'\nAnchors missing L6M Y ({len(missing_l6m)}):')
    print(missing_l6m[['anchor_id', 'anchor_name', 'n_months_l6m',
                       'l6m_first_month', 'l6m_last_month']]
          .head(10).to_string(index=False))

# L6M window distribution among modeling anchors — surfaces the 5 stale-reporting stores
print(f'\nDistribution of l6m_last_month across the 186 anchors:')
print(cov['l6m_last_month'].dt.to_period('M').value_counts().sort_index(ascending=False)
      .head(10).to_string())

# Correlation between the two targets
both = cov.dropna(subset=['monthly_sales_avg_alltime_jpy', 'monthly_sales_avg_l6m_jpy'])
print(f'\nCorrelation (post-ramp alltime vs personal L6M, n={len(both)}):')
print(f'  Pearson  = {both[["monthly_sales_avg_alltime_jpy","monthly_sales_avg_l6m_jpy"]].corr().iloc[0,1]:.3f}')
print(f'  Spearman = {both[["monthly_sales_avg_alltime_jpy","monthly_sales_avg_l6m_jpy"]].corr(method="spearman").iloc[0,1]:.3f}')

# Growth signal: L6M / all-time per store
ratio = (both['monthly_sales_avg_l6m_jpy'] / both['monthly_sales_avg_alltime_jpy']).rename('l6m_vs_alltime')
print(f'\nL6M / all-time ratio (>1 means recent stronger than lifetime average):')
print(ratio.describe(percentiles=[.1, .5, .9]).round(2).to_string())

Anchors                                                            : 186
  with all-time Y (post-ramp, >= 2 months)                          : 186
  with L6M Y (personal last 6 post-ramp, >= 2 months)              : 186

Anchors with thin sample (1-5 post-ramp months — newly opened, treat with caution):
anchor_id                     anchor_name  open_year  n_months_alltime l6m_first_month l6m_last_month
  gc_1541                   LaLaport ANJO     2025.0                 3      2025-10-01     2025-12-01
  gc_1534             AEON MALL TSUMINAMI     2025.0                 4      2025-09-01     2025-12-01
  gc_1535               minamoa HIROSHIMA     2025.0                 4      2025-09-01     2025-12-01
  gc_1355        Sagamiono station SQUARE     2025.0                 4      2025-09-01     2025-12-01
  gc_1545      FASHION CRUISE HITACHINAKA     2025.0                 3      2025-10-01     2025-12-01
  gc_1357               Harajuku Jingumae     2025.0                 4      2025-09

### What's done vs pending

**Methodology — current**
- Catchment = **`CATCHMENT_SIDE_M` × `CATCHMENT_SIDE_M` square** centered on the anchor (default 500m).
  Change `CATCHMENT_SIDE_M` in Section 1 to scale to other sizes (250 / 1000 / …).
- **All distance / count features computed inline in Python** — no Alteryx dependency.
  - In-catchment count: geopandas `sjoin` against axis-aligned squares.
  - 1 / 3 / 5 km ring counts: vectorized haversine on raw POI tables (~0.5M ops, <100 ms).
  - Nearest-POI (cannibalization): same haversine matrix, `argmin` on each row.
  - Size-weighted sums (e.g. SC sales, station passengers) reuse the same square + ring masks.

**Done in this notebook**
- Anchors: existing-store anchors (open > `MIN_OPEN_YEARS`) with single-square catchment + `store_area_m2` / `store_area_sqft` / `seats` attached from Gongcha xlsx
- POI tables standardized: Starbucks, Tully's, tea-brand competitors (ESRI-geocoded), combined competitors, Gongcha-self, SC, schools (overall + university / high_school / vocational slices), stations (S12 with 2019 & 2024 passenger time series), Shinkansen subset
- Uniform `build_block()` pipeline → 500m-sq + 1/3/5km rings (+ optional nearest, + optional size sums) for every POI source
- ESRI enrich wired against the 500m square catchment (population / age-sex / income / consumption)
- Admin spatial join: anchor → prefecture / municipality / density tier
- Shinkansen flags: catchment-level (within 500m / 1km / 3km / 5km, plus nearest distance) AND municipality-level
- Cannibalization signal: `nearest_gongcha_dist_km` per anchor (computed inline)
- MLIT 1km mesh foot traffic ingested + cached (pickle) with anchor-bbox prefilter; spatially overlaid onto anchors as `foot_traffic_*` features (residential / commuter / leisure proxies for 2019 + 2021)
- Two-stage feature table: `model_features.csv` (everything except foot traffic, saved at 8b) → `model_features_full.csv` (with foot traffic, saved at 8d)
- **Y targets (Section 10)**: `monthly_sales_avg_alltime_jpy` and `monthly_sales_avg_l6m_jpy` written to `output/tables/y_targets.csv` (consumed by `04_modeling.ipynb` Section 4)

**Pending**
- Whitespace grid mode (Section 2b) only stubbed; run when shifting from "evaluate existing" to "score new candidates". Foot-traffic block (8d) already has anchor-chunking via `CHUNK_SIZE_FT=5000` so 100K+ candidates won't blow memory.